# <center>Harness Engineering 驾驭工程基础入门 · 第二节：手搓 Mini Harness</center>

&emsp;&emsp;在第一节里，你建立了 Harness Engineering 的认知地图——**八大故障模式**是问题，**八大核心机制**是解法骨架，**三支柱**是评价标准。第一节刻意留了两条故障模式（⑥缺权限闸、⑧成本失控）没有主解，本节要把它们闭环——所以你今天拿到的是 **11 个机制（8 核心 + 3 扩展）** 的完整 Mini Harness，而不是第一节骨架的原样复刻。今天的目标只有一件事：<font color=red>把这张认知地图变成真实可运行的代码</font>。你将从零搭建 11 个文件、11 个机制，每个机制都扣住一条故障模式，写完一个就在清单上打一个勾。

&emsp;&emsp;这不是玩具项目。课程结束时，你将亲眼看到一次完整的 E2E（端到端）demo：一个配备了全部 11 个机制的 Harness agent，在真实任务上跑完 7 个步骤，产生 15053 个 tokens，留下一份落盘的 `progress.md`——对比单轮的 Baseline（1104 tokens）。这两个数字背后的差距，就是本节要诚实回答的核心问题：Harness 值不值、值在哪里。

&emsp;&emsp;接下来你会按以下主线展开：先用 3 分钟重新点亮第一节的认知地图（第零章），建立今日的文件地图（第一章），然后逐一实现全部 11 个机制——第二章到第八章覆盖第一节已讲过的 8 个核心机制（翻译成代码）、第九章补齐三个扩展机制（Permission Gate / Hooks / Token Budget），最终在第十章串联成一次完整 E2E demo，课程末尾（第十一章）指向下一步。

> &emsp;&emsp;📅 **时效性说明**：本课件基于 2026 年 4 月技术状态编写，使用 `deepseek-chat` 模型（DeepSeek 直连）验证 E2E demo。依赖版本锁定（已在 `mini_harness/requirements.txt` 中 pin）：`openai==2.26.0`、`python-dotenv==1.2.2`、`pytest==9.0.3`、`httpx==0.28.1`。OpenAI Python SDK 兼容 OpenRouter / DeepSeek 等 OpenAI 协议端点。API 定价、模型名称等时效性信息以官方为准。

> &emsp;&emsp;**学员画像与前置要求**：本课程面向**具备 Python 基础、已完成第一节学习**的开发者。前置要求：熟悉 `openai` Python SDK 基本用法、了解 LLM Function Calling 概念、会配置 `.env` 环境变量。学完本课后，你将能够独立实现一个含 **11 个机制（8 核心 + 3 扩展）** 的 Mini Harness，对照第一节的八大故障模式清单解释每个机制的价值，并诚实评估 Harness 的真实代价。

---

## <center>第零章：上节回顾 + 今日目标</center>

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231636411.png" width=50% alt="11 机制全景路线图"></p>


&emsp;&emsp;上节课你认识了一个核心事实：Agent 不是智能实体，而是一个由代码组成的、可以反复调用 LLM 的循环。这个循环本身非常脆弱——它会循环失控、会吞掉工具报错、会在很长的会话中把自己搞傻、会无声无息地烧掉你的钱。这些脆弱点被归纳成八大故障模式，Harness 的全部价值就是系统性地应对它们。

&emsp;&emsp;第一节给你拆了**八大核心机制**（Agent Loop / Tool Use / Progress Tracking / Context Management / Feature List / Verification Loop / Subagents / Generator-Evaluator），但它刻意只闭环了其中 6 条故障模式（①②③④⑤⑦），把 **⑥缺权限闸** 和 **⑧成本失控** 留给本节——所以本节在 8 核心机制之外，还要补 3 个扩展机制：**Permission Gate（解 ⑥）、Token Budget（解 ⑧）、Hooks（贯穿所有机制的事件挂钩基础设施）**。合起来就是本节要交付的 **11 个机制 = 8 核心 + 3 扩展**。

&emsp;&emsp;今天你要从"知道"跨越到"能做"。在你开始写第一行代码之前，先建立全景印象——下面是你手边这份 Mini Harness 代码库和故障模式/机制的对应关系：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>八大故障模式 × 11 个机制（8 核心 + 3 扩展）× 代码文件对照</font></p>
<div class="center">

| 机制编号 | 机制名称 | 救的故障模式 | 实现文件 | 本节模块 |
|---------|---------|---------|---------|---------|
| ① | Agent Loop | ① 循环失控 | core.py | 第二章 |
| ② | Tool Use | ④ tool 错误吞 | core.py | 第二章 |
| ③ | Progress Tracking | ⑤ 状态丢失 | progress.py | 第三章 |
| ④ | Context Management | ② context 溢出 / ③ cache miss | context.py | 第四章 |
| ⑤ | Feature List | ② context 溢出（源头防控） | planner.py | 第五章 |
| ⑥ | Verification Loop | ④ tool 错误吞（下游） | verifier.py | 第六章 |
| ⑦ | Subagents | ② context 溢出 / ⑧ 成本失控 | subagent.py | 第七章 |
| ⑧ | Generator-Evaluator | ⑦ 缺自动化评审 | evaluator.py | 第八章 |
| — | *— 以上为第一节八大核心机制 · 以下为本节三个扩展机制 —* | — | — | — |
| ⑨ | Permission Gate | ⑥ 缺权限闸 | permission.py | 第九章 |
| ⑩ | Hooks | 贯穿所有机制的事件挂钩基础设施 | hooks.py | 第九章 |
| ⑪ | Token Budget | ⑧ 成本失控 | budget.py | 第九章 |

</div>

&emsp;&emsp;今天你的产出目标：`mini_harness/` 目录下的这 11 个文件（含 1 个配置中心 + 10 个机制模块文件）全部理解、能读懂每一行关键代码、知道 `run_e2e_demo.py` 是如何把它们串联起来的。**120 分钟后你会获得两件事**：一是独立搭一套 11 机制 Mini Harness 的能力；二是对照八大故障模式清单诊断任意 Agent 项目的眼力——看到一个新的 agent 产品，你能立刻说出它解了哪几条故障模式、没解哪几条、生产环境上会踩哪几个坑。

---

## <center>第一章：项目骨架与环境</center>

&emsp;&emsp;第零章 已经把 11 个文件和 11 个机制的对应关系摆在你面前。在逐一进入每个机制之前，你需要先在磁盘上找到它们——这就是 第一章 的任务：建立整体文件地图，搭好环境，让后续所有机制都能从这个基础上运行起来。

&emsp;&emsp;在进入任何一个机制之前，你需要先建立整体文件地图。本节<font color=red>刻意不用 LangChain / LangGraph / OpenAI Agents SDK / Anthropic Agent SDK</font>——目的是让你看清裸骨架：每一块肌肉是什么、做什么、怎么连接。框架会帮你隐藏复杂度，但代价是你不知道出了问题该去哪里找。这个 Mini Harness 不为省事，为看懂。

### 1.1 目录结构

&emsp;&emsp;`mini_harness/` 的实际目录结构如下图所示：整个项目按职责划分为 `harness/`（含 1 个配置中心 + 10 个机制模块文件，承载全部 11 个机制）、`scripts/`（端到端演示脚本 `run_e2e_demo.py`）、`test_data/`（E2E 任务素材，含 `refactor_project/main.py` 折扣计算重复代码 case）三个顶层目录。其中 `harness/` 下的 11 个 `.py` 文件就是课程主线——`config.py` 是所有机制共享的配置中心，`core.py` 承载机制①②（Agent Loop + Tool Use），其余 9 个文件每一个对应一个机制（`permission.py` / `hooks.py` / `budget.py` 是本节新增的三个机制）。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231643157.png" width=50% alt="mini_harness 目录结构示意图">
</div>

&emsp;&emsp;记住这条规律：<font color=red>11 个文件 = 1 个配置中心 + 10 个机制模块文件（承载 11 个机制）</font>。`config.py` 是所有机制的共享配置；`core.py` 一个文件承载两个机制（① Agent Loop + ② Tool Use）；剩余 9 个文件各自对应一个机制。

### 1.2 环境搭建

&emsp;&emsp;你需要先完成以下环境准备，才能运行后续的任何代码示例。`mini_harness/` 目录已提供 `requirements.txt`，直接安装：

In [5]:
!python --version

Python 3.11.15


In [ ]:
!pip install -r requirements.txt -q

&emsp;&emsp;安装完成后，在项目根目录（`HarnessEngineering/`）创建 `.env` 文件，填入以下内容：

```ini
# DeepSeek 直连（推荐，价格低、无代理）
DEEPSEEK_API_KEY=your_deepseek_api_key_here

# 或者走 OpenRouter 路由（支持多模型切换）
OPENROUTER_API_KEY=your_openrouter_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
```

> **【安全提醒】** `.env` 文件包含 API Key，切勿提交到 Git 仓库。建议在 `.gitignore` 添加 `.env`。

&emsp;&emsp;加载环境变量的标准方式：

In [6]:
from dotenv import load_dotenv
import os

# 从 HarnessEngineering/.env 加载凭证
load_dotenv()

# 读取 API Key（DeepSeek 直连或 OpenRouter）
api_key = os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENROUTER_API_KEY")
base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

&emsp;&emsp;以上代码加载了 `.env` 文件中的凭证，并根据优先级选择 DeepSeek 或 OpenRouter。你后续所有 demo 代码都基于这套配置。

In [7]:
import json
from openai import OpenAI

client = OpenAI()

# 第一部分：定义工具函数
def add(a: int, b: int) -> int:
    """将两个整数相加"""
    return a + b

def get_weather(city: str) -> str:
    """查询城市天气（Mock 数据）"""
    return f"{city}：晴天，25°C"

# 工具注册表
TOOLS = {
    "add": add,
    "get_weather": get_weather,
}

# 第二部分：手写 tool schema（让学员感受这个痛点）
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "将两个整数相加",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "第一个数"},
                    "b": {"type": "integer", "description": "第二个数"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询城市天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名"},
                },
                "required": ["city"],
            },
        },
    },
]

# 第三部分：Agent 循环核心
def run_agent(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]

    round_num = 0
    while True:
        round_num += 1
        print(f"\n===== 第 {round_num} 轮 LLM 调用 =====")

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        msg = response.choices[0].message
        messages.append(msg)

        # 没有工具调用 → 任务完成
        if not msg.tool_calls:
            print(f"最终回答：{msg.content}")
            return msg.content

        # 执行所有工具调用
        for call in msg.tool_calls:
            name   = call.function.name
            args   = json.loads(call.function.arguments)
            result = TOOLS[name](**args)
            print(f"  调用工具: {name}({args}) => {result}")

            messages.append({
                "role":         "tool",
                "tool_call_id": call.id,
                "content":      str(result),
            })

# 测试运行
run_agent("北京今天天气怎么样？另外 88 加 99 等于多少？")



===== 第 1 轮 LLM 调用 =====
  调用工具: get_weather({'city': '北京'}) => 北京：晴天，25°C
  调用工具: add({'a': 88, 'b': 99}) => 187

===== 第 2 轮 LLM 调用 =====
最终回答：北京今天天气晴天，温度为25°C。88加99等于187。


'北京今天天气晴天，温度为25°C。88加99等于187。'

### 1.3 配置中心：HarnessConfig

&emsp;&emsp;所有 11 个机制共享一个配置对象 `HarnessConfig`，避免将参数散落在各处。来看它的完整实现：

In [8]:
from dataclasses import dataclass
from typing import Optional
import os


# HarnessConfig 是所有机制共享的配置数据类，dataclass 自动生成 __init__
@dataclass
class HarnessConfig:
    """Harness agent 核心配置，所有机制共享同一实例。"""

    max_steps: int = 50            # Agent Loop 最大步数（故障模式①刹车）
    max_tokens: int = 100000       # 会话 token 上限（软上限，硬上限由 BudgetGuard 控制）
    context_threshold: int = 12000 # Context 压缩触发阈值（估算 token 数）
    # 默认走 OpenRouter 路由名；可通过环境变量 MODEL_NAME 覆盖
    model_name: str = "deepseek/deepseek-chat"
    api_key: Optional[str] = None  # API Key（从 env 注入）
    base_url: Optional[str] = None # API endpoint
    temperature: float = 0.7       # 生成随机性

    @classmethod
    def from_env(cls) -> "HarnessConfig":
        """从环境变量构建配置，自动读取 OPENROUTER_API_KEY 和 MODEL_NAME。"""
        # 优先读 MODEL_NAME，未设置则用默认 OpenRouter 路由前缀格式
        return cls(
            model_name=os.getenv("MODEL_NAME", "deepseek/deepseek-chat"),
            api_key=os.getenv("OPENROUTER_API_KEY"),
            base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
        )

&emsp;&emsp;`HarnessConfig` 是一个 Python `dataclass`，字段有合理默认值，`from_env()` 类方法从环境变量构建实例。注意 `max_steps=50` 这个字段——它是机制①（Agent Loop）防止循环失控的硬刹车，你会在 第二章 里看到它被 `while steps < config.max_steps` 引用。

&emsp;&emsp;需要注意的是，`from_env()` 当前只读取 `OPENROUTER_API_KEY`。如果你在 `.env` 里配置的是 `DEEPSEEK_API_KEY`，不能直接用 `from_env()`，需要显式构造：

In [9]:
import os
from mini_harness.harness.config import HarnessConfig

# DeepSeek 直连方式：显式传入 api_key 和 base_url
# 可通过环境变量 MODEL_NAME 覆盖模型名（此处为默认值示例）
config = HarnessConfig(
    api_key=os.getenv("DEEPSEEK_API_KEY"),    # 读取 DeepSeek Key
    base_url="https://api.deepseek.com",       # DeepSeek 直连端点
    model_name=os.getenv("MODEL_NAME", "deepseek-chat"),  # DeepSeek 直连格式；走 OpenRouter 时改为 "deepseek/deepseek-chat"
)

&emsp;&emsp;两种方式都能跑通 E2E demo：`from_env()` 适合 OpenRouter 路由，显式构造适合 DeepSeek 直连。选一个与你的 `.env` 配置对应的方式即可。

---

## <center>第二章：机制①② Agent Loop + Tool Use</center>

&emsp;&emsp;从 第一章 的文件地图到实际执行，第一步是理解 `core.py`。这是整个 Mini Harness 的心脏——Agent Loop 和 Tool Use 都住在这里。没有这两个机制，后续所有机制都无从谈起：Progress Tracking 要挂载的 hook 事件，Permission Gate 要拦截的工具调用，Token Budget 要累计的 token 数——全都发生在这个主循环里。

&emsp;&emsp;在进入 2.1 之前，先集中准备第二章～第九章所有教学节选共用的标准 import 与 LLM 客户端。把下面这个 cell 放进 Notebook 的第一个代码位置执行一次，后续每个小节的节选代码都会默认共享这些名字；如果你漏跑这一 cell，后续节选里出现的 `Path` / `client` / `json` 等符号会抛 `NameError`。

In [10]:
# 第二章～第九章所有教学节选共用的标准 import（Notebook 第一次执行时运行本 cell）
import os
import json
import datetime
import inspect
import subprocess
import re
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

# OpenAI 兼容客户端（默认走 DeepSeek 直连；如走 OpenRouter，把 base_url 改为 https://openrouter.ai/api/v1）
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

# 从 mini_harness 直接导入真实实现里的辅助符号（教学节选中会引用这些名字）
from mini_harness.harness.hooks import HookManager, NoOpHooks
from mini_harness.harness.core import _safe_usage_ints,_build_tool_schemas,dispatch_tool
# 9.4 节三机制组合接入会用到下面三个类（先在共用 cell 拉齐，避免后续 NameError）
from mini_harness.harness.progress import ProgressTracker
from mini_harness.harness.permission import PermissionGate
from mini_harness.harness.budget import BudgetGuard

&emsp;&emsp;这个 cell 完成三件事：一是把后续节选会用到的标准库（`json / datetime / inspect / subprocess / re / pathlib / typing`）一次性拉齐；二是创建全局 `client`，后续每个 `client.chat.completions.create(...)` 直接引用它；三是从 `mini_harness/harness/` 导入真实实现里的工具类（`HookManager` / `NoOpHooks`）和内部 helper（`_safe_usage_ints`），让节选里出现的名字都有实体可以绑定。下面每个节选展示的是"这个函数/类长什么样"，真实生产用请直接 `from mini_harness.harness.<module> import <符号>`。

### 2.1 Agent Loop 的四相

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231636434.png" width=50% alt="Agent Loop 四相流程"></p>


&emsp;&emsp;Agent Loop 的核心结构可以用四个词概括：**Gather Context → Verify → Take Action → Iterate**。这四相构成一个闭环，每一轮结束时要么继续（Iterate），要么停止（返回最终答案）。下面是 `core.py` 的主循环节选：

In [11]:
def run_agent(
    user_goal: str,
    tools,
    config,
    system_prompt=None,
    hooks=None,
    budget=None,
) -> dict:
    """
    Agent 主循环。四相：Gather Context → Verify → Take Action → Iterate。

    Args:
        user_goal: 用户任务描述
        tools: 可调用工具字典，key=工具名，value=可调用函数
        config: HarnessConfig 实例
        system_prompt: 可选的系统提示词（Verification Loop 用）
        hooks: 可选的 HookManager（Progress/Permission/Budget 挂载点）
        budget: 可选的 BudgetGuard（成本硬上限）

    Returns:
        dict，包含 answer / steps / tokens / errors / budget_report / blocked_tools
    """
    # hooks 为 None 时退化为 NoOpHooks，向后兼容原版六机制行为
    if hooks is None:
        hooks = NoOpHooks()

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_goal})

    tool_schemas = _build_tool_schemas(tools)  # 自动从函数签名生成 JSON Schema
    steps = 0
    errors = []
    total_tokens = 0

    # 触发 session_start 事件（Progress Tracker 在这里写入 progress.md 头部）
    hooks.trigger("session_start", user_goal)

    while steps < config.max_steps:   # 故障模式①刹车：max_steps 硬上限
        steps += 1
        hooks.trigger("pre_iteration", steps, messages)

        # Gather Context：调用 LLM 获取下一步决策
        response = client.chat.completions.create(
            model=config.model_name,
            messages=messages,
            tools=tool_schemas if tool_schemas else None,
            temperature=config.temperature,
        )

        # 累计 token 消耗，触发 Budget 检查
        usage = getattr(response, "usage", None)
        in_tokens, out_tokens = _safe_usage_ints(usage)
        if budget is not None:
            budget.add(in_tokens, out_tokens)   # 超限抛 BudgetExceeded（core 的 try/except 捕获后优雅终止）

        choice = response.choices[0]
        message = choice.message

        # Verify：finish_reason == stop 且无工具调用 → 任务完成
        if not message.tool_calls:
            answer = message.content or ""
            hooks.trigger("session_stop", answer, {"steps": steps, "tokens": total_tokens})
            return {"answer": answer, "steps": steps, "tokens": total_tokens, "errors": errors}

        # === Take Action：先把 assistant message 追加到历史（关键！）===
        # tool_calls 必须原样回写，否则下一轮 tool result 无法关联 tool_call_id
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in message.tool_calls
            ] if message.tool_calls else None,
        })

        # 然后才是逐个执行工具
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            # Permission Gate 挂在这里（gate 事件，可 veto）
            ok, reason = hooks.trigger_gate("pre_tool_use", tool_name, tool_args)
            if not ok:
                messages.append({"role": "tool", "tool_call_id": tool_call.id,
                                  "content": json.dumps({"error": reason})})
                continue

            # 执行工具，结构化错误回传（不吞掉）
            result = dispatch_tool(tool_name, tool_args, tools)
            hooks.trigger("post_tool_use", tool_name, tool_args, result)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    # Iterate：未在 max_steps 内完成则返回失败
    return {"answer": "Max steps reached without completion", "steps": steps, "tokens": total_tokens, "errors": errors}

&emsp;&emsp;这段不足 100 行的代码，构成了整个 Harness（驾驭工程）的 **“大心脏”**。它没有依赖任何第三方 Agent 框架，而是使用纯原生 Python 实现了一个完整的 **Agent 四相循环（Gather → Verify → Action → Iterate）**。

&emsp;&emsp;为了真正看懂底层的运转机制，请直接对照上方代码中体现的这**四个核心阶段**来阅读：

&emsp;&emsp;**相一：Gather Context（收集信息）**
    这是向大模型发起通信的环节。在执行 `client.chat.completions.create` 时，系统将当前的完整对话历史（`messages`）以及规范化后的工具 JSON 描述清单（`tool_schemas`），一同通过 API 发送给大模型，获取下一步的任务决策。

&emsp;&emsp;**相二：Verify（验证退出）**
    这是判定任务是否真正结束的**唯一出口**。当流程走到 `if not message.tool_calls` 判定时，如果大模型的返回结果中不再包含任何调用工具的请求，只是一段普通文本，系统就判定大模型认为任务已完成（或已无法继续）。此时，Agent 立即终止当前的循环，截断并返回最终答案。

&emsp;&emsp;**相三：Take Action（采取行动）**
    **【深坑预警区】** 如果大模型返回了调用工具的请求，系统将进入此阶段。请极度注意这里的**代码执行顺序**，一旦弄错会导致全盘崩溃：

1. **必须先记录指令**：即代码中的 `messages.append({"role": "assistant"...})`，必须先把大模型下发的“要求调用某工具”这条消息追加进历史对话中。

    2. **再执行工具与反馈**：即随后的 `for tool_call in message.tool_calls:`，循环遍历去真正执行具体的 Python 工具代码。无论工具执行成功还是抛出错误，都**不能把错误吞掉**，而是把返回结果封装成 JSON 字符串，再次推入对话历史（`role: "tool"`）。

*   **为什么顺序不能改？** 这是 API 接口的强契约要求。如果下一轮大模型分析历史记录时，只看到了工具的最终运行结果（`role: "tool"`），却找不到之前是哪条指令要求运行它的（`role: "assistant"`），API 会立即抛出 `tool_call_id mismatch (ID不匹配)` 错误！

&emsp;&emsp;**相四：Iterate（安全迭代门禁）**
    在这个无限循环的外围，这套骨架代码密布着用于保障生产环境安全的**硬刹车机制**：
    *   **步数刹车（故障模式①防线）**：外层的 `while steps < config.max_steps` 确保了如果大模型由于代码错误或逻辑缺陷陷入原地打转的死循环，只要到达设定的步数上限，系统将强制跳出。
    *   **预算门禁（故障模式⑧防线）**：在调用 `budget.add()` 的地方，每次收到大模型回复后，系统会立刻计算此轮消耗的 Token 并换算成计费。一旦本次循环导致累计消费数突破了硬性上限，它会立刻阻断运行，防止引发无底洞般的资金损耗。

### 2.1.E 验证测试（mock）

&emsp;&emsp;本节用 mock LLM 客户端演示 Agent Loop 的循环结构与退出条件，不演示真实模型推理质量。两个对照 case：正常 2 轮完成 vs. max_steps 硬刹车，让你亲眼看到循环如何被强制终止。

In [12]:
# === 【验证-MOCK】机制 2.1：4 相循环 + max_steps 硬刹车 ===
# 用假 LLM 客户端演示循环结构与退出条件，不真调 API。真实运行看第十章 E2E demo。

import json
from unittest.mock import patch
from types import SimpleNamespace
from mini_harness.harness.core import run_agent
from mini_harness.harness.config import HarnessConfig

def make_fake_response(text=None, tool_name=None, tool_args=None,
                       tool_id="fake_tc_1", prompt_tokens=80, completion_tokens=40):
    if tool_name:
        tc = SimpleNamespace(id=tool_id, type="function",
            function=SimpleNamespace(name=tool_name, arguments=json.dumps(tool_args or {})))
        msg = SimpleNamespace(content=None, tool_calls=[tc])
    else:
        msg = SimpleNamespace(content=text or "done", tool_calls=None)
    return SimpleNamespace(
        choices=[SimpleNamespace(message=msg, finish_reason="stop")],
        usage=SimpleNamespace(prompt_tokens=prompt_tokens, completion_tokens=completion_tokens),
    )

class FakeOpenAIClient:
    """按预设序列返回 fake response，模拟 LLM 调用。"""
    def __init__(self, responses):
        self.responses = responses
        self.call_count = 0
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        idx = min(self.call_count, len(self.responses) - 1)
        self.call_count += 1
        return self.responses[idx]

def echo(text: str) -> str:
    return f"echo: {text}"

# Case 1：正常完成（2 轮）——第 1 轮 tool_call，第 2 轮 final answer
print("=== Case 1: 正常完成（2 轮退出）===")
resp_c1 = [
    make_fake_response(tool_name="echo", tool_args={"text": "hello"}),
    make_fake_response(text="任务完成，echo 结果已返回"),
]
config1 = HarnessConfig(max_steps=5, api_key="fake", base_url="http://fake")
with patch("mini_harness.harness.core.OpenAI", return_value=FakeOpenAIClient(resp_c1)):
    r1 = run_agent("echo hello", {"echo": echo}, config1)
print(f"  steps={r1['steps']}  errors={r1['errors']}")
print(f"  answer={r1['answer']}")

# Case 2：max_steps 硬刹车——LLM 永远返回 tool_call，永不给 final answer
print("\n=== Case 2: max_steps 硬刹车（永不终止的 LLM）===")
resp_c2 = [make_fake_response(tool_name="echo", tool_args={"text": "loop"})]
config2 = HarnessConfig(max_steps=3, api_key="fake", base_url="http://fake")
with patch("mini_harness.harness.core.OpenAI", return_value=FakeOpenAIClient(resp_c2)):
    r2 = run_agent("echo forever", {"echo": echo}, config2)
print(f"  steps={r2['steps']}  errors={r2['errors']}")
print(f"  answer={r2['answer']}")

=== Case 1: 正常完成（2 轮退出）===
  steps=2  errors=[]
  answer=任务完成，echo 结果已返回

=== Case 2: max_steps 硬刹车（永不终止的 LLM）===
  steps=3  errors=[]
  answer=Max steps reached without completion


&emsp;&emsp;**关键观察**：Case 1 的 steps=2 证明循环在"无 tool_calls"时正确退出（Verify 相命中）；Case 2 的 steps=3 与 answer="Max steps reached" 证明 max_steps 是真硬刹车——即使 mock LLM 永远不给 final answer，run_agent 也在第 3 轮强制返回。真实 LLM 跑出来会有 token 消耗和推理延迟，mock 演示的是循环结构本身；真实 LLM 调用看第十章 E2E demo（包含真实 token 消耗 + 真实输出）。

&emsp;&emsp;看完循环骨架被硬刹车截停之后，你接下来要解决的是循环里**工具调用失败的可见性**问题——这就是 Tool Use 机制要做的事。

### 2.2 Tool Use：schema 契约 + 结构化错误回传

&emsp;&emsp;Tool Use 机制由两部分组成：一是 LLM 如何知道可以调用什么工具（schema 契约），二是工具调用失败时如何处理（结构化错误回传而不是吞掉）。先看 schema 生成：

In [13]:
import inspect
import json

def _build_tool_schemas(tools: dict) -> list:
    """
    从工具函数签名自动生成 OpenAI 格式的 tool schema。

    旧版本 properties/required 都为空，LLM 不知道传哪些参数。
    本版本用 inspect.signature 从函数签名抽取参数名、类型、是否必填。

    Args:
        tools: 工具字典，key=工具名，value=可调用函数

    Returns:
        list，每个元素是 OpenAI function calling 格式的 schema dict
    """
    # Python 类型 → JSON Schema 类型的映射表
    # LLM 只认识 JSON Schema 类型，不认识 Python 原生类型
    _PY_TO_JSON = {str: "string", int: "integer", float: "number",
                   bool: "boolean", list: "array", dict: "object"}

    schemas = []

    for name, func in tools.items():
        # 读取函数的 docstring 作为工具描述，告诉 LLM 这个工具"能做什么"
        doc = (func.__doc__ or "").strip()

        properties = {}  # 存放参数的 JSON Schema 描述
        required = []    # 存放必填参数名列表

        # inspect.signature(func) 通过反射机制解析函数签名
        # 返回一个 Signature 对象，包含所有参数的名称、类型注解、默认值等信息
        # 例如 def add(a: int, b: int = 0) → 能拿到 a、b 的完整元信息
        sig = inspect.signature(func)

        # sig.parameters 是一个有序字典：{参数名: Parameter 对象}
        for param_name, param in sig.parameters.items():

            # 跳过 *args（VAR_POSITIONAL）和 **kwargs（VAR_KEYWORD）
            # 这类可变参数无法映射为固定的 JSON Schema 字段
            if param.kind in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD):
                continue

            # param.annotation 是参数的类型注解（如 int、str）
            # 若未写类型注解则为 inspect.Parameter.empty，默认降级为 "string"
            json_type = _PY_TO_JSON.get(param.annotation, "string")

            # 将参数描述写入 properties，这是 OpenAI function calling 的标准字段
            properties[param_name] = {
                "type": json_type,
                "description": f"参数 {param_name}",
            }

            # param.default 是参数的默认值
            # inspect.Parameter.empty 是一个特殊标记，表示"没有默认值"
            # 没有默认值 → 调用时必须传 → 加入 required 列表
            if param.default is inspect.Parameter.empty:
                required.append(param_name)

        # 按照 OpenAI function calling 的标准格式拼装 schema
        schemas.append({
            "type": "function",
            "function": {
                "name": name,
                "description": doc or f"Tool: {name}",
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required,
                },
            },
        })

    return schemas


In [14]:
# ── 三个有代表性的测试工具函数 ────────────────────────────────────
def get_weather(city: str, unit: str = "celsius") -> str:
    """查询指定城市的实时天气信息。"""
    ...
def calculator(a: float, b: float, op: str) -> float:
    """执行基础四则运算，op 支持 +/-/*/。"""
    ...
def send_email(to: str, subject: str, body: str, cc: list = None) -> bool:
    """发送电子邮件。cc 为可选的抄送列表。"""
    ...
# ── 执行并格式化输出 ──────────────────────────────────────────────
tools = {
    "get_weather": get_weather,
    "calculator":  calculator,
    "send_email":  send_email,
}
schemas = _build_tool_schemas(tools)
print(json.dumps(schemas, ensure_ascii=False, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "查询指定城市的实时天气信息。",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "参数 city"
          },
          "unit": {
            "type": "string",
            "description": "参数 unit"
          }
        },
        "required": [
          "city"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "执行基础四则运算，op 支持 +/-/*/。",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "number",
            "description": "参数 a"
          },
          "b": {
            "type": "number",
            "description": "参数 b"
          },
          "op": {
            "type": "string",
            "description": "参数 op"
          }
        },
        "required": [
          "a",
          

&emsp;&emsp;请注意，大语言模型并不能直接看懂 Python 的函数源码，它只认标准的 JSON Schema。

&emsp;&emsp;**旧版痛点**：过去，开发者只需告诉模型工具有哪些（但 `properties` 未详细定义），导致模型在使用时全靠“瞎蒙”参数，极易引发幻觉和调用崩溃。或者需要开发者手工维护另一份几百行的 JSON 配置文件，如果忘记更新，极易导致代码与配置脱节。

&emsp;&emsp;**Harness 的解法（自动反射翻译）**：`_build_tool_schemas` 引入了 Python 的 `inspect.signature` 机制。它在系统启动时，使用“运行时反射”直接透视内存中的函数对象结构。不仅自动提取了 `param_name`（参数缩写），还抓取了 `param.annotation`（通过映射把 `str` 翻译为 JSON 的 `string`）以及 `param.default`（借此判断是否无默认值，从而精准推入 `required` 必填数组）。

&emsp;&emsp;**核心收益**：依靠该机制实现了“业务源码”与“调用契约文档”的 **100% 自动同源同步**。你在 Python 里增删一个参数，给大模型的 JSON 契约瞬间自适应改变，极其优雅地解决了“大模型漏传错传参数”的底层顽疾。

&emsp;&emsp;再看结构化错误回传——这是机制②救故障模式④（tool 错误吞）的关键：

In [15]:
def dispatch_tool(name: str, args: dict, tools: dict) -> str:
    """
    执行工具并返回 JSON 字符串结果（成功或结构化错误）。

    Args:
        name: 工具名
        args: 工具参数字典
        tools: 可用工具字典

    Returns:
        JSON 字符串，成功时是工具返回值，失败时是 {"error": "原因"}
    """
    if name not in tools:
        # 工具不存在：返回结构化错误而不是抛异常
        return json.dumps({"error": f"Tool '{name}' not found"})
    try:
        result = tools[name](**args)
        if not isinstance(result, str):
            result = json.dumps(result)
        return result
    except Exception as e:
        # 执行失败：错误信息回传给 LLM，让它自动重试或换策略
        return json.dumps({"error": f"Tool execution failed: {str(e)}"})

&emsp;&emsp;这段仅 20 行的代码，是整个 Harness 系统能够实现 **“自动自我纠偏与修复”** 的核心枢纽。

&emsp;&emsp;**传统开发的盲区**：在普通的 Python 业务开发中，遇到程序报错通常只有两种做法：一是让程序直接抛出异常崩溃（但这会导致 Agent 立刻死机，前功尽弃丢失长程对话）；二是用 `try...except: pass` 强行吞掉异常当作无事发生（但这会导致大模型产生严重幻觉，以为工具执行成功了，从而基于错误假象继续瞎编结果）。这两条路在 Agent 开发里都是死路。

&emsp;&emsp;**Harness 的解法（结构化安全气囊）**：`dispatch_tool` 既坚决不让主系统崩溃，也绝不吞掉异常。它将捕获到的 Python 报错信息包裹进一个 `{"error": "具体的报错原因（如：Division by zero）"}` 的 JSON 字符串中，然后将其**作为执行结果原封不动地回传**，写入到大模型的历史上下文中。

&emsp;&emsp;**核心收益**：大语言模型具备极强的代码阅读与纠错能力。当它在下一轮交互中看到返回的竟然包含 `error` 提示时，它会像一名真正的人类研发一样醒悟（“原来是我参数路径给错了”），然后自觉修正参数重新发起调用请求。这种**绝不吞报错**的结构化回传机制，赋予了 Agent 在复杂任务中软着陆并自我治愈的能力。

### 2.2.E 验证测试

&emsp;&emsp;演示 `dispatch_tool` 如何将"工具不存在"与"执行异常"统一转为结构化错误字符串返回，而不是让 agent 崩溃或静默吞掉异常。

In [17]:
# === 【验证】机制 2.2：错误回传 vs 错误吞掉 ===
# 纯本地，无需 LLM API。预期 1 秒内跑完。

import json
from mini_harness.harness.core import dispatch_tool

# 注册一个会触发除零异常的工具
def divide(a, b):
    return json.dumps({"result": a / b})

tools = {"divide": divide}

# Case 1：LLM 编造了不存在的工具名
r1 = dispatch_tool("nonexistent_tool", {}, tools)
print("【Case 1】工具不存在：")
print(" ", r1)
print()

# Case 2：工具存在但执行抛异常（除以零）
r2 = dispatch_tool("divide", {"a": 1, "b": 0}, tools)
print("【Case 2】工具抛异常：")
print(" ", r2)
print()

# Case 3：正常调用，返回结果
r3 = dispatch_tool("divide", {"a": 10, "b": 2}, tools)
print("【Case 3】工具成功：")
print(" ", r3)

# 清理注册（避免污染后续 cell）
del tools, divide

【Case 1】工具不存在：
  {"error": "Tool 'nonexistent_tool' not found"}

【Case 2】工具抛异常：
  {"error": "Tool execution failed: division by zero"}

【Case 3】工具成功：
  {"result": 5.0}


&emsp;&emsp;**关键观察**：三个 case 全部返回 JSON 字符串而非抛出异常——错误被"显性化"成结构化数据，agent 可以读取 `error` 字段决策重试或上报，而不会静默失败。Case 2 尤其重要：`ZeroDivisionError` 被 `dispatch_tool` 拦截转换，说明工具层的防护是系统性的，不依赖每个工具自己处理异常。

&emsp;&emsp;工具失败已经能被显性化了。最后你可能担心：换一个 SDK 是不是要把 `dispatch_tool` 重写？答案在下一节。

---

## <center>第三章：机制③ Progress Tracking + Memory 三层导读</center>

&emsp;&emsp;第二章 的 Agent Loop 解决了"怎么跑"的问题。但有一个问题 Loop 本身无法解决：如果任务跑到一半崩溃了，重新启动时 agent 什么都不记得，下一个子 agent 也不知道之前做了什么。这是故障模式⑤（状态丢失）。机制③ Progress Tracking 是你手里解决这个问题的最小基础设施。

### 3.1 Progress Tracking 的实现

&emsp;&emsp;你给 `ProgressTracker` 准备的就是这三件事：每次 session 开始时写入头部、每次 tool 调用后追加一条记录、session 结束时写入摘要。`progress.py` 的核心类 `ProgressTracker` 通过三个 hook 事件把会话过程记录到 `progress.md` 文件：

In [18]:
import json
import datetime
from pathlib import Path

class ProgressTracker:
    """
    进度追踪器，通过 HookManager 与 core.run_agent 解耦。

    设计思路：
      不直接修改 run_agent 代码，而是把日志逻辑"挂"在三个事件点上。
      run_agent 触发事件 → HookManager 分发 → ProgressTracker 响应 → 写入 progress.md

    三个挂载点：
      - session_start：写入会话开始分隔线和目标
      - post_tool_use：每次工具调用后写入一条日志
      - session_stop：写入会话结束摘要（步骤数/tokens/最终答案）
    """

    def __init__(self, path: str = "progress.md", max_content_len: int = 200):
        """
        Args:
            path: progress.md 存储路径（绝对或相对路径均可）
            max_content_len: 每条日志最大字符数，防止工具结果撑爆文件
        """
        # Path() 将字符串路径转为 pathlib.Path 对象
        # pathlib 比 os.path 更现代，支持 / 拼接、mkdir、open 等链式操作
        self.path = Path(path)

        # 单条日志的最大字符数上限，超出部分截断并加 "..."
        self.max_content_len = max_content_len

        # 用当前时间戳生成唯一 session ID（格式：20240424_152300）
        # 每次 run_agent 实例化一个新 ProgressTracker，session_id 自动不同
        # 多次运行的日志追加在同一文件里，靠 session_id 区分彼此
        self.session_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        # 工具调用计数器，每触发一次 post_tool_use 事件自增 1
        # 用于在日志里标记 step 1 / step 2 / step 3 ...
        self._entry_count = 0

    def register_to(self, hooks) -> None:
        """
        将三个 handler 批量挂载到 HookManager，一行代码完成接入。

        调用方只需：
            tracker = ProgressTracker()
            tracker.register_to(agent.hooks)   ← 一行搞定

        不需要改动 run_agent 内部任何代码，实现"零侵入"接入。
        """
        # "session_start" 事件触发时，调用 self.on_session_start
        hooks.register("session_start", self.on_session_start)
        # "post_tool_use" 事件触发时，调用 self.on_post_tool_use
        hooks.register("post_tool_use", self.on_post_tool_use)
        # "session_stop" 事件触发时，调用 self.on_session_stop
        hooks.register("session_stop",  self.on_session_stop)

    def on_session_start(self, user_goal: str = "") -> None:
        """
        session 开始时触发（由 HookManager 在 run_agent 入口处调用）。
        写入一条带时间戳和目标预览的分隔线，标记新会话的开始。
        """
        # isoformat(timespec="seconds") 输出格式：2024-04-24T15:23:00
        ts = datetime.datetime.now().isoformat(timespec="seconds")

        # 对用户目标也做截断，防止超长 prompt 撑爆日志
        goal_preview = str(user_goal)[:self.max_content_len]

        # 用 Markdown 格式写入，方便后续在编辑器里直接阅读
        content = (
            f"\n---\n"
            f"## Session {self.session_id} started at {ts}\n"
            f"- goal: {goal_preview}\n"
        )
        self._append(content)

    def on_post_tool_use(self, tool_name="", tool_args=None, result="") -> None:
        """
        每次工具调用完成后触发（由 HookManager 在工具执行后调用）。
        写入结构化日志：时间戳、步骤序号、工具名、入参、返回值。
        """
        # 自增步骤计数，记录这是本次 session 的第几次工具调用
        self._entry_count += 1
        ts = datetime.datetime.now().isoformat(timespec="seconds")

        # json.dumps 将参数字典序列化为字符串，ensure_ascii=False 保留中文
        # tool_args 为 None 时用空字典兜底，避免 json.dumps(None) 报错
        args_str = json.dumps(tool_args or {}, ensure_ascii=False)

        # 对工具返回结果做截断：
        # 超出 max_content_len 的部分用 "..." 替代，防止搜索/读文件结果撑满日志
        result_str = (
            str(result)[:self.max_content_len]
            + ("..." if len(str(result)) > self.max_content_len else "")
        )

        # 拼装 Markdown 三级标题格式的日志条目
        content = (
            f"\n### {ts} · step {self._entry_count} · tool={tool_name}\n"
            f"- args: {args_str[:self.max_content_len]}\n"
            f"- result: {result_str}\n"
        )
        self._append(content)

    def on_session_stop(self, answer="", metrics=None) -> None:
        """
        session 结束时触发（由 HookManager 在 run_agent 返回前调用）。
        写入本次会话的总结：总步骤数、消耗 tokens、最终回答摘要。
        """
        ts = datetime.datetime.now().isoformat(timespec="seconds")

        # metrics 由 run_agent 传入，包含 steps / tokens 等统计数据
        # 用 or {} 兜底，避免调用方忘记传时报 TypeError
        metrics = metrics or {}

        # 最终回答也做截断，只记录前 max_content_len 个字符
        answer_preview = str(answer)[:self.max_content_len]

        content = (
            f"\n## Session {self.session_id} ended at {ts}\n"
            # dict.get(key, default) 取值，key 不存在时返回 "?" 而不是报 KeyError
            f"- steps: {metrics.get('steps', '?')}\n"
            f"- tokens: {metrics.get('tokens', '?')}\n"
            f"- final answer: {answer_preview}\n"
            f"---\n"
        )
        self._append(content)

    def _append(self, content: str) -> None:
        """
        将内容追加写入 progress.md（私有方法，供三个 handler 统一调用）。

        两个关键点：
          1. mkdir(parents=True, exist_ok=True)：目录链不存在时自动创建，exist_ok 避免重复创建报错
          2. open("a")：追加模式，不覆盖历史记录；若用 "w" 则每次运行都会清空文件
        """
        # 父目录不存在时自动递归创建（例如 logs/2024/progress.md）
        self.path.parent.mkdir(parents=True, exist_ok=True)

        # "a" = append 追加模式，历史日志永久保留
        # encoding="utf-8" 确保中文不乱码
        with self.path.open("a", encoding="utf-8") as f:
            f.write(content)


&emsp;&emsp;当你把 `register_to(hooks)` 这一行加进 `run_agent` 的初始化代码里，这个 agent 的每一步行为就开始留痕了。`register_to` 是 `ProgressTracker` 的核心 API——用一行代码把三个 handler 挂到 `HookManager`，之后每次 tool call 都会自动触发 `on_post_tool_use` 写入日志。`ProgressTracker` 不需要知道 `run_agent` 内部的实现，`run_agent` 也不需要知道 `ProgressTracker` 的存在——这是 Hooks 机制（第九章 会详细讲）的价值所在。

### 3.1.E 验证测试

&emsp;&emsp;通过对比"挂载前"与"挂载后"两个场景，直接观察 ProgressTracker 注册到 HookManager 后 4 次事件触发如何转化为磁盘上真实存在的 progress.md 内容。

In [19]:
# ================================================================
# 【验证】机制 3.1：ProgressTracker 落盘测试
# 纯本地，无需 LLM API，预期 2 秒内完成
# ================================================================

import os
import tempfile
from pathlib import Path
from mini_harness.harness.progress import ProgressTracker

def print_section(title: str):
    """输出带分隔线的标题，方便在 notebook 里区分各测试块"""
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

def show_file(path: str, max_lines: int = 10):
    """读取并打印文件内容，超出 max_lines 时截断"""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    lines = [l for l in lines if l.strip()]          # 过滤空行
    for line in lines[:max_lines]:
        print(f"  {line}")
    if len(lines) > max_lines:
        print(f"  ... （共 {len(lines)} 行，已截断）")


# ---------------------------------------------------------------
# Case 1：不挂 ProgressTracker → trigger 后文件应保持为空
# ---------------------------------------------------------------
print_section("Case 1：不挂 ProgressTracker，文件应保持为空")

tmp1 = tempfile.NamedTemporaryFile(suffix=".md", delete=False)
tmp1.close()

hooks_bare = HookManager()
hooks_bare.trigger("post_tool_use",
                   tool_name="search",
                   tool_args={"q": "hello"},
                   result="42")

size = os.path.getsize(tmp1.name)
status = "正确：无落盘" if size == 0 else f"异常：文件大小 {size} bytes"
print(f"  文件大小: {size} bytes  →  {status}")
os.unlink(tmp1.name)


# ---------------------------------------------------------------
# Case 2：挂载 ProgressTracker → 4 次 trigger 后真实落盘
# ---------------------------------------------------------------
print_section("Case 2：挂载 ProgressTracker，验证落盘内容")

# 创建临时文件路径（不预先写入，交给 ProgressTracker 自行创建）
tmp2_path = "progress_test.md"

hooks = HookManager()
tracker = ProgressTracker(path=tmp2_path, max_content_len=80)
tracker.register_to(hooks)   # 一行接入：自动挂载 3 个事件

# 模拟一次完整的 Agent 会话生命周期
hooks.trigger("session_start",
              user_goal="抓取并汇总 10 篇 LLM 论文")

hooks.trigger("post_tool_use",
              tool_name="search",
              tool_args={"q": "LLM agent 2024"},
              result="找到 10 篇相关论文")

hooks.trigger("post_tool_use",
              tool_name="read_file",
              tool_args={"path": "paper1.pdf"},
              result="Abstract: We propose a new agent framework...")

hooks.trigger("session_stop",
              answer="已汇总 10 篇论文，核心结论如下...",
              metrics={"steps": 2, "tokens": 512})

# 验证结果
all_lines = [l for l in Path(tmp2_path).read_text().splitlines() if l.strip()]
print(f"  落盘总行数: {len(all_lines)}  →  {'有内容' if all_lines else '无内容'}")
print(f"\n  📄 progress.md 内容预览：")
show_file(tmp2_path)

# ---------------------------------------------------------------
# 清理
# -----------------------------------------
# ----------------------
# print_section("清理临时文件")
# os.unlink(tmp2_path)
# print(" 临时文件已删除")



  Case 1：不挂 ProgressTracker，文件应保持为空
  文件大小: 0 bytes  →  正确：无落盘

  Case 2：挂载 ProgressTracker，验证落盘内容
  落盘总行数: 15  →  有内容

  📄 progress.md 内容预览：
  ---
  ## Session 20260424_195405 started at 2026-04-24T19:54:05
  - goal: 抓取并汇总 10 篇 LLM 论文
  ### 2026-04-24T19:54:05 · step 1 · tool=search
  - args: {"q": "LLM agent 2024"}
  - result: 找到 10 篇相关论文
  ### 2026-04-24T19:54:05 · step 2 · tool=read_file
  - args: {"path": "paper1.pdf"}
  - result: Abstract: We propose a new agent framework...
  ## Session 20260424_195405 ended at 2026-04-24T19:54:05
  ... （共 15 行，已截断）


&emsp;&emsp;**关键观察**：Case 1 文件恒为 0 字节，说明 hook 事件本身不产生任何副作用；register_to 仅一行，却让后续 4 次 trigger 全部转化为磁盘写入，这就是 agent 崩溃后能从断点续传的物理基础——progress.md 始终在磁盘上，重启后读 read_recent() 即可接手。

&emsp;&emsp;看到落盘的最小骨架后，下面是 Full Harness 在真实任务里跑出的 `progress.md` 节选——同一个 `ProgressTracker`，规模放大到 7 步追踪。

### 3.3 Memory 三层架构导读

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231639216.png" width=50% alt="Memory 三层架构"></p>


&emsp;&emsp;`progress.md` 解决的是 Memory 最底层——事实层。生产级 Memory 其实是**三层递进架构**，复杂度随问题域增长。为了让你对 Harness 的 Memory 有完整图景，下面这张表把三层同时列出来对比：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Memory 三层架构 × Mini Harness 实现位置</font></p>
<div class="center">

| 层级 | 存储什么 | Mini Harness 实现 | 生产级实现 | 救的故障模式 |
|------|---------|-------------------|-----------|---------|
| 事实层 | 完成了什么步骤、使用了哪些工具 | `progress.md` 追加写 | `CLAUDE.md` + `MEMORY.md` | ⑤ 状态丢失 |
| 历史层 | 跨会话的操作记录，可检索 | 未实现 | SQLite + FTS5 全文检索 | ⑤ 状态丢失（进阶） |
| 画像层 | 用户偏好、项目风格、重复告知 | 无 | Honcho 辩证建模 | 体验提升 |

</div>

&emsp;&emsp;Mini Harness 只实现了最底层的事实层——这是教学侧刻意的选择：先让你看到最小可用版本怎么工作。生产级 agent（如 Claude Code）会把三层一起做：事实层保证不忘、历史层保证可查、画像层保证不重复解释。你在第一节的生态地图里看到的 `Cline` / `Roo Code` / `Aider` 等工具，都是在这三层上做工程取舍的产物。

---

## <center>第四章：机制④ Context Management + 三铁律 + Cache/Rot</center>

&emsp;&emsp;你现在已经有了一个能跑多步的 Loop，能落盘进度的 Tracker。但还有一个隐患：随着 session 越来越长，messages 列表会无限增长。一旦超过模型的 context 窗口，整个 Loop 就会崩溃。更微妙的是，即使没崩，过多的历史信息也会稀释有效信号——agent 越跑越"傻"。这两个问题合称故障模式②（context 溢出）和故障模式③（cache miss）。机制④ Context Management 是应对它们的工具箱。

### 4.1 压缩策略：head + tail + middle summary

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231636453.png" width=50% alt="Context 压缩策略"></p>


&emsp;&emsp;`context.py` 的 `compress_if_needed` 函数实现了一个简洁的三段保护策略：

In [20]:
from mini_harness.harness.context import compress_if_needed, estimate_tokens,_summarize_middle

def compress_if_needed(messages: list, threshold: int = 12000) -> list:
    """
    当消息列表的估算 token 数超过阈值时触发压缩。

    压缩策略（三段保护）：
      head：保留前 2 条（system prompt + 原始用户任务，不可丢失）
      tail：保留最近 6 条（当前推理窗口，不可丢失）
      middle：压缩为摘要（工具调用统计 + 关键工具名）

    Args:
        messages: 完整消息历史列表
        threshold: 触发压缩的估算 token 阈值

    Returns:
        压缩后的消息列表（head + summary_msg + tail）
    """
    token_count = estimate_tokens(messages)
    # 未超阈值：直接返回原列表，不做任何修改
    if token_count < threshold:
        return messages

    # 消息数量太少时不压缩，避免把有效 context 全部清空
    if len(messages) <= 8:
        return messages

    head = messages[:2]    # system + user（绝对不裁）
    tail = messages[-6:]   # 最近 6 条（保住当前推理上下文）
    middle = messages[2:-6]  # 中间部分压缩为摘要

    # 把 middle 压缩为统计摘要（工具名 + 消息条数）
    summary_content = _summarize_middle(middle)
    summary_msg = {"role": "system", "content": f"[Context Summary]\n{summary_content}"}

    return head + [summary_msg] + tail

&emsp;&emsp;三段策略的逻辑很清晰：头部两条（`system prompt` + 原始用户任务）绝对不能丢——那是 LLM 理解整个会话的根基；尾部六条是当前推理的上下文——丢了就断档；中间部分是已经完成的历史操作，用摘要替换不会影响当前推理，但能大幅减少 token 占用。

&emsp;&emsp;token 估算用的是一个简化方案：

In [21]:
def estimate_tokens(messages: list) -> int:
    """
    简化 token 估算：总字符数 / 4。

    Args:
        messages: 消息列表

    Returns:
        估算 token 数（整数）

    注意：这是教学级估算，生产场景应使用 tiktoken（OpenAI）
    或 anthropic.count_tokens() 做精确计数。
    """
    total_chars = 0
    # 遍历每条消息，累加 content 和 tool_calls 的字符总量
    for msg in messages:
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", "")
        total_chars += len(str(content)) + len(str(tool_calls))
    # 除以 4 得到粗略估算（英文约 4 字符/token，中文约 1.5-2 字符/token）
    return total_chars // 4

&emsp;&emsp;`chars / 4` 的估算对英文文本误差约 10-15%，对中文会低估（中文约 1.5-2 字符/token）。这在教学场景够用，但生产环境要用 `tiktoken` 做精确计数，避免在临界点频繁触发或完全不触发压缩。

### 4.1.E 验证测试

&emsp;&emsp;用短/长两组消息对比 `compress_if_needed` 的实际行为：短消息不触发压缩原样返回，长消息触发后头尾锁定、中间变摘要，亲眼看到压缩比。

In [22]:
# === 【验证】机制 4.1：长 vs 短 messages 压缩比对照 ===
# 纯本地，无需 LLM API。预期 1 秒内跑完。

from mini_harness.harness.context import compress_if_needed, estimate_tokens,_summarize_middle

# --- Case 1：短消息，不触发压缩 ---
short_msgs = [
    {"role": "system", "content": "你是一个助手。"},
    {"role": "user",   "content": "你好"},
]
tokens_short = estimate_tokens(short_msgs)
out_short    = compress_if_needed(short_msgs)

print("=== Case 1：短消息（不触发压缩）===")
print(f"  估算 tokens : {tokens_short}")
print(f"  原始条数    : {len(short_msgs)}")
print(f"  压缩后条数  : {len(out_short)}")
print(f"  是否原样返回: {out_short is short_msgs}")
print()

# --- Case 2：长消息（10 条），触发压缩 ---
long_msgs = [
    {"role": "system", "content": "你是一个助手。"},
    {"role": "user",   "content": "帮我完成一个长期任务。"},
]
for _ in range(8):                                # 中间 8 条 assistant，各 8000 字符
    long_msgs.append({"role": "assistant", "content": "x" * 8000})

tokens_before = estimate_tokens(long_msgs)
out_long      = compress_if_needed(long_msgs)
tokens_after  = estimate_tokens(out_long)
ratio         = tokens_after / tokens_before

print("=== Case 2：长消息（触发压缩）===")
print(f"  压缩前 tokens : {tokens_before}")
print(f"  压缩前条数    : {len(long_msgs)}")
print(f"  压缩后条数    : {len(out_long)}")
print(f"  压缩后 tokens : {tokens_after}")
print(f"  压缩比         : {ratio:.2%}")
print(f"  压缩后结构    : {[m['role'] for m in out_long]}")

=== Case 1：短消息（不触发压缩）===
  估算 tokens : 2
  原始条数    : 2
  压缩后条数  : 2
  是否原样返回: True

=== Case 2：长消息（触发压缩）===
  压缩前 tokens : 16004
  压缩前条数    : 10
  压缩后条数    : 9
  压缩后 tokens : 12054
  压缩比         : 75.32%
  压缩后结构    : ['system', 'user', 'system', 'assistant', 'assistant', 'assistant', 'assistant', 'assistant', 'assistant']


&emsp;&emsp;**关键观察**：Case 1 tokens 仅为 2，远低于阈值 12000，函数直接原样返回同一对象（`is` 为 True）。Case 2 触发压缩后，10 条变 9 条（head 2 + summary 1 + tail 6），中间 2 条 assistant 被合并为一条统计摘要；压缩比约 75%，说明三段保护策略在极端"中间全是重复内容"场景下的真实效果。

&emsp;&emsp;你已经看到压缩在 token 维度的效果。但 token 数只是 context 管理的一面，还有一个看不见的维度叫 Prompt Cache——它有三条铁律。

### 4.2 三铁律：Prompt Cache 的工程约束

&emsp;&emsp;故障模式③（cache miss）的来源是 Prompt Cache 的工作机制：LLM 服务商对稳定的前缀做缓存（Anthropic 有 `cache_control`，OpenAI 有 automatic prompt caching），相同前缀的请求命中缓存可以降低延迟和成本。但 cache 有 TTL，而且只要前缀改变就立即失效。

&emsp;&emsp;因此，在 agent 运行过程中，有三件事是<font color=red>绝对禁止中途修改</font>的：

> **【Context Management 三铁律】** （以下三条基于 Prompt Cache 前缀一致性原理推导，灵感参考 [AGENTS.md 规范](https://agents.md/) 的 Context 管理相关内容）：
> 1. **不许中途改 tools 定义**——toolset 与 system prompt 共同构成 Prefix Cache 的 key，中途修改会导致缓存前缀失效，已缓存的 token 全部作废。
> 2. **不许中途重载 memory**——重载会插入新内容，改变前缀，打破缓存
> 3. **不许中途重建 system prompt**——同上，前缀任何改变都会让缓存失效

&emsp;&emsp;这三条规律背后的原理相同：<font color=red>Prompt Cache 看的是前缀的完整一致性</font>。只要前缀有任何改变，缓存就视为不命中，下一次请求要重新计算整个前缀——在长会话中，这意味着额外的延迟和成本。违反铁律（比如中途重建 system prompt）会导致 cache miss：同样的请求，响应时间从约 200ms 涨到约 1200ms，成本也随之上升；遵守铁律保证前缀稳定，相同前缀命中缓存后重复计算的输入 token 不重复计费。

### 4.3 Context Rot：大窗口不等于更好

&emsp;&emsp;现代大模型的 context 窗口越来越大（1M tokens 已经不罕见），但这不意味着"塞越多越好"。<font color=red>Context Rot（上下文腐烂）</font>描述了一个真实现象：session 越长，agent 越傻——不是模型能力下降了，而是旧信息、重复日志、过期决策把有效信号淹没了，模型在噪音里找不到重点。

&emsp;&emsp;应对 Context Rot 有三条工具：

- **阶段性压缩**：`compress_if_needed` 定期清理中间历史（本节机制④）

- **计划跟踪**：`todo_tool` 用结构化清单替代"用 LLM 记任务"（本节机制⑤）

- **渐进式披露**：子任务只传必要上下文，不暴露父任务的全部历史（本节机制⑦ Subagents）

&emsp;&emsp;**你现在掌握了**：用 `compress_if_needed` 在阈值处压缩 context；用三铁律保证 Prompt Cache 命中；识别 Context Rot 的症状并用三条工具反制。接下来进入机制⑤ Feature List——看看怎么从任务源头切小、防止 context 一开始就爆炸。

---

## <center>第五章：机制⑤ Feature List</center>

&emsp;&emsp;第四章 的压缩是"发现 context 溢出后的补救"，机制⑤ Feature List 是"防止 context 溢出在源头发生"。它的核心思路是：<font color=red>用结构化任务清单控制 agent 的注意力</font>，让它每步只做一件事，而不是试图一口气吞下整个任务。

### 为什么需要 Feature List？

当我们把一个复杂任务直接丢给 Agent，它会把**所有子任务的背景、工具调用、中间结果**全部堆进同一个 context 窗口。随着对话轮次增加，context 迅速膨胀，最终触发第四章的压缩机制——但压缩是有代价的：**信息有损、缓存失效、推理质量下降**。

Feature List 的逻辑是：**任务越小，context 越干净；context 越干净，Agent 越专注。**


### 工作机制

Feature List 将一个大目标拆解为若干**原子化的可执行条目（Feature）**，每个 Feature 满足以下约束：

<div style="display: flex; justify-content: center;">

| 约束 | 说明 |
|:----:|------|
| **单一职责** | 每条 Feature 只做一件具体的事 |
| **可验证** | 完成后有明确的输出或状态变更 |
| **顺序解耦** | 前后 Feature 之间依赖关系明确，避免隐式耦合 |

</div>

Agent 每次只从清单中取出**当前 Feature**，处理完毕后标记完成、取下一条，始终保持 context 聚焦。


### 5.1 todo_tool 三模式

&emsp;&emsp;`planner.py` 只有一个函数 `todo_tool`，但它支持三种不同的操作模式：

In [23]:
# 全局 TODO 列表（module 级变量，在单次运行中跨工具调用持久）
TODOS = []


def todo_tool(todos=None, merge: bool = False) -> str:
    """
    管理 agent 的结构化任务清单。

    三种调用模式：
      1. 查询模式：todos=None → 返回当前清单（不修改）
      2. 替换模式：todos=[...], merge=False → 完全替换清单（任务初始化时用）
      3. 合并模式：todos=[...], merge=True → 更新或追加条目（更新状态时用）

    Args:
        todos: todo 条目列表，每条格式 {"id": str, "content": str, "status": "pending|in_progress|completed"}
               None 表示只查询不修改
        merge: True=合并模式，False=替换模式

    Returns:
        当前 TODO 清单的 JSON 字符串
    """
    global TODOS

    if todos is None:
        # 查询模式：返回当前清单
        return json.dumps(TODOS, ensure_ascii=False, indent=2)

    if merge:
        # 合并模式：按 id 更新已有条目，追加新条目
        existing_ids = {t["id"] for t in TODOS}
        for todo in todos:
            if todo["id"] in existing_ids:
                # 找到匹配 id 的条目，原位更新
                for i, existing in enumerate(TODOS):
                    if existing["id"] == todo["id"]:
                        TODOS[i] = todo
                        break
            else:
                TODOS.append(todo)  # 新条目直接追加
    else:
        # 替换模式：完全覆盖
        TODOS = todos

    return json.dumps(TODOS, ensure_ascii=False, indent=2)

&emsp;&emsp;三种模式对应 agent 任务生命周期的三个阶段：任务开始时用替换模式创建清单（`merge=False`），执行过程中用合并模式更新状态（`merge=True`，把 `pending` 改为 `in_progress` 再改为 `completed`），任务完成前用查询模式检查是否全部完成（`todos=None`）。

> **【Notebook 重复执行警告】** 上面的 `todo_tool` 使用了模块级 `global TODOS`。在 Notebook 里多次执行同一个替换模式（`merge=False`）的 cell 时，这个全局状态可能还残留上一次的 todos，让你看到"清单怎么没清空"的疑惑。如果清单看起来不对，**重启 kernel** 或先显式调用一次 `reset_todos()`（`planner.py` 提供的清零 helper），是最快的修复方式。这个陷阱在教学 Notebook 里频率比生产环境高——生产每次跑都是新进程，不会有残留。

### 5.1.E 验证测试

&emsp;&emsp;演示 `todo_tool` 的三种调用模式：替换模式初始化任务列表，合并模式按 id 更新或追加条目，查询模式读取终态快照，覆盖任务生命周期全流程。

In [24]:
# ================================================================
# 【验证】机制 5.1：todo_tool 状态机三种模式
# 纯本地，无需 LLM API，预期 1 秒内完成
# ================================================================

import json
from mini_harness.harness.planner import todo_tool, reset_todos

# 状态符号映射，方便肉眼快速识别
STATUS_ICON = {
    "pending":     "⬜ pending",
    "in_progress": "🔄 in_progress",
    "completed":   "✅ completed",
}

def show_todos(raw_json: str, title: str):
    """格式化打印 todo 列表"""
    print(f"\n{'─'*48}")
    print(f"  {title}")
    print(f"{'─'*48}")
    for t in json.loads(raw_json):
        icon = STATUS_ICON.get(t["status"], t["status"])
        print(f"  [{t['id']}] {icon:<20}  {t['content']}")


# ── 初始化：防止重复执行污染 ────────────────────────────────────
reset_todos()

# ================================================================
# 模式一：替换模式（merge=False）—— 完全覆盖，任务初始化时使用
# ================================================================
r1 = todo_tool(
    todos=[
        {"id": "1", "content": "收集需求", "status": "pending"},
        {"id": "2", "content": "编写代码", "status": "pending"},
        {"id": "3", "content": "测试验证", "status": "pending"},
    ],
    merge=False,
)
show_todos(r1, "模式一：替换模式 ─ 初始化 3 条任务")

# ================================================================
# 模式二：合并模式（merge=True）—— 更新已有条目
# id=1 存在 → 原位替换状态为 in_progress
# ================================================================
r2 = todo_tool(
    todos=[{"id": "1", "content": "收集需求", "status": "in_progress"}],
    merge=True,
)
show_todos(r2, "模式二：合并模式 ─ 更新 id=1 → in_progress")

# ================================================================
# 模式二（续）：合并模式 —— 追加新条目
# id=4 不存在 → 追加到清单末尾
# ================================================================
r3 = todo_tool(
    todos=[{"id": "4", "content": "撰写文档", "status": "pending"}],
    merge=True,
)
show_todos(r3, "模式二（续）：合并模式 ─ 追加新 id=4")

# ================================================================
# 模式三：查询模式（todos=None）—— 只读，返回终态快照
# ================================================================
r4 = todo_tool(todos=None)
show_todos(r4, "模式三：查询模式 ─ 终态快照（只读）")

# ── 尾部清零，不污染下游 cell ────────────────────────────────────
reset_todos()
final = json.loads(todo_tool(todos=None))
print(f"\n{'─'*48}")
print(f"  reset 后 TODOS 长度: {len(final)}  →  {'已清空' if not final else '未清空'}")
print(f"{'─'*48}")



────────────────────────────────────────────────
  模式一：替换模式 ─ 初始化 3 条任务
────────────────────────────────────────────────
  [1] ⬜ pending             收集需求
  [2] ⬜ pending             编写代码
  [3] ⬜ pending             测试验证

────────────────────────────────────────────────
  模式二：合并模式 ─ 更新 id=1 → in_progress
────────────────────────────────────────────────
  [1] 🔄 in_progress         收集需求
  [2] ⬜ pending             编写代码
  [3] ⬜ pending             测试验证

────────────────────────────────────────────────
  模式二（续）：合并模式 ─ 追加新 id=4
────────────────────────────────────────────────
  [1] 🔄 in_progress         收集需求
  [2] ⬜ pending             编写代码
  [3] ⬜ pending             测试验证
  [4] ⬜ pending             撰写文档

────────────────────────────────────────────────
  模式三：查询模式 ─ 终态快照（只读）
────────────────────────────────────────────────
  [1] 🔄 in_progress         收集需求
  [2] ⬜ pending             编写代码
  [3] ⬜ pending             测试验证
  [4] ⬜ pending             撰写文档

────────────────────────────────────

&emsp;&emsp;**关键观察**：替换模式是任务初始化的唯一正确入口，一次写入覆盖全量；合并模式按 id 做增量更新，找到 id 则覆盖、找不到则追加，正好对应"任务推进"和"临时加塞"两种场景；终态快照证明 id=1 的 status 已从 `pending` 升为 `in_progress`，而 id=2/3 保持不变——这正是状态机精确推进而非全量覆盖的核心价值。

&emsp;&emsp;三模式的小例子跑通了。下面是 Full Harness 在 E2E demo 里 7 步任务中 todo 状态机的真实轨迹——你会看到机制⑤在生产任务里的完整使用模式。

### 5.2 E2E（端到端） demo 中的真实运行轨迹

&emsp;&emsp;来自 `demo_outputs/progress.md`（2026-04-22 真实跑通），你可以看到 agent 在 7 个步骤中的 todo 状态机变化：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>E2E Demo 中 Feature List 状态机变化轨迹</font></p>
<div class="center">

| Step | 操作 | todo 1 状态 | todo 2 状态 | todo 3 状态 |
|------|------|------------|------------|------------|
| 1 | 替换模式创建清单 | pending | pending | pending |
| 2 | 合并模式更新 | in_progress | pending | pending |
| 3 | refactor_code 工具调用 | in_progress | pending | pending |
| 4 | 合并模式更新 | completed | in_progress | pending |
| 5 | 合并模式更新 | completed | completed | in_progress |
| 6 | 合并模式更新 | completed | completed | completed |
| 7 | 输出最终答案（无工具调用） | — | — | — |

</div>

&emsp;&emsp;7 步中有 5 步（步骤 1、2、4、5、6）是 todo 操作。这不是低效，而是 Feature List 机制的刻意设计：<font color=red>每步只做一件事的纪律，是防止 context 溢出的源头防控</font>。不用 Feature List 的 agent 会试图一步解决全部问题，生成一大段分析然后迭代修改——这既不可追踪，也很容易因为"想太多"导致输出质量下降。

### 5.3 Mini Harness vs 生产级 Feature List

&emsp;&emsp;`todo_tool` 是 Feature List 机制的最简实现。与生产级 agent 相比：Claude Code 使用 200+ 项结构化清单，支持多小时任务规划；OpenAI Codex 通过 `PLANS.md` 文件持久化任务清单（跨 session 保留）。Mini Harness 的 `todo_tool` 是 in-memory 的，重启就清零——这是它被称为"玩具版"的原因，同时也足够教学目的。


### 生产级 vs 教学级的关键差异

<div style="display: flex; justify-content: center;">

| 维度 | Mini Harness `todo_tool` | 生产级 Agent |
|:----:|:----:|:----:|
| **存储方式** | Python 内存（重启清零） | 文件 / 数据库（跨 session 持久） |
| **清单规模** | 数条～十几条 | Claude Code 支持 200+ 项 |
| **状态粒度** | `pending / in_progress / completed` | 支持优先级、依赖关系、截止时间 |
| **并发支持** | 单线程顺序执行 | 支持子任务并行、阻塞等待 |
| **可观测性** | print 输出 | 结构化日志 + 可视化看板 |

</div>



&emsp;&emsp;生产级实现的复杂性来自**工程需求**（容错、持久化、并发），而不是**核心机制**。`todo_tool` 去掉了所有工程噪音，让你能清晰地看到 Feature List 的**本质**：

> **用一个外部数据结构承载任务状态，把"全局规划的复杂性"从 LLM 的 context 里剥离出去。**

掌握这个思想后，把 `TODOS = []` 换成数据库、把 `json.dumps` 换成持久化文件，就是生产级的实现路径。**机制相同，规模不同。**


### 从 `todo_tool` 到生产级的演进路径

**① Mini Harness（本课）**
- 存储：`TODOS = []` 内存变量，重启即清零
- 适合：教学演示、单次任务

**② 第一步升级：持久化**
- 存储：写入 `PLANS.md` 或 `todos.json` 磁盘文件
- 优点：跨 session 保留，Agent 重启后可续接任务
- 代表：OpenAI Codex 的 `PLANS.md` 方案

**③ 第二步升级：结构化**
- 存储：SQLite 或任务数据库
- 优点：支持查询、过滤、优先级排序、依赖关系
- 代表：Claude Code 的 200+ 项结构化清单

**④ 第三步升级：并发**
- 存储：异步任务队列（如 Celery、Ray）
- 优点：子任务并行执行，大幅缩短多步骤任务耗时
- 代表：生产级多 Agent 协作系统

> 💡 **核心不变的是机制**：无论哪个阶段，本质都是"把任务状态外置到 LLM context 之外"。演进的只是存储介质和工程能力，不是设计思想。


---

## <center>第六章：机制⑥ Verification Loop</center>

&emsp;&emsp;在第二章里你看到了结构化错误回传：工具执行失败时，`dispatch_tool` 把错误包成 JSON 传回给 LLM。但这只是"错误不被吞掉"，还不是"错误被真正验证和修复"。故障模式④（tool 错误吞）的下游后果是：agent 生成了代码，认为自己做完了，但代码根本跑不通。机制⑥ Verification Loop 的核心是：<font color=red>用真实工具验证结果，而不只是让 LLM 自己说"我检查了"</font>。


### 问题的根源：LLM 的自我验证是不可信的

LLM 在生成内容之后，如果被问"你确定这是对的吗？"，它倾向于回答"是的，我检查过了"。这不是谎言，而是模型的统计特性——它无法真正"运行"自己生成的代码，只能根据上下文判断答案看起来是否合理。

这意味着：**LLM 的自我确认没有验证效力**。一个代码片段只有被真实执行过，才算被验证过。



### 与第二章错误回传的本质区别

<div style="display: flex; justify-content: center;">

| 机制 | 触发时机 | 验证主体 | 作用 |
|:----:|:----:|:----:|:----:|
| 第二章：结构化错误回传 | 工具调用本身出错时 | 工具执行层 | 确保错误不被吞掉 |
| 机制⑥：Verification Loop | 工具调用成功后 | 独立验证工具 | 确保结果本身是正确的 |

</div>

二者解决的是不同层次的问题：前者是"工具有没有跑起来"，后者是"工具跑出来的结果对不对"。

### 核心设计原则

验证必须由工具完成，而不是由 LLM 完成。LLM 适合做的是：分析失败原因、生成修复方案。工具适合做的是：执行验证、返回客观结论。将这两种能力组合成闭环，才是 Verification Loop 的完整形态。

### 6.1 两层验证架构

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231647227.png" width=50% alt="两层验证架构"></p>


&emsp;&emsp;`verifier.py` 实现了两层验证：

&emsp;&emsp;**第一层（引导层）**：通过 `SYSTEM_PROMPT_WITH_VERIFICATION` 把验证思维植入 agent 的系统提示词，让它在完成每步后自问"我是否需要验证？"：

In [31]:
# 引导层 system prompt：把自我验证思维嵌入 agent 的行为规范
# 使用位置：run_agent(system_prompt=SYSTEM_PROMPT_WITH_VERIFICATION)
SYSTEM_PROMPT_WITH_VERIFICATION = """
    你是一个具备工具调用能力的 AI 助手。

    在解决任务时，请遵循以下行为规范：
    1. 将复杂问题分解为若干具体步骤
    2. 调用可用工具获取真实信息，不依赖推测
    3. 在得出最终结论前，主动验证中间结果
    4. 对不确定的内容，通过追加工具调用进行二次核查

    验证清单（每步完成后自检）：
    - 所需信息是否已全部通过工具获取？
    - 当前结论是否有工具返回结果作为依据？
    - 是否存在需要额外工具调用来确认的假设？
    - 最终答案是否完整、准确？

    准确性优先于速度，宁可多调用一次工具，也不输出未经验证的结论。
    """


&emsp;&emsp;这个 system prompt 在 E2E demo 的 `run_full_harness` 中被传入 `run_agent`（参见 `scripts/run_e2e_demo.py` 的 `SYSTEM_PROMPT_WITH_VERIFICATION` 引用）。引导层的局限性在于：它只是让 LLM "想到验证"，但 LLM 可以自我声明"我检查过了"而不实际执行——因此需要第二层。

&emsp;&emsp;**第二层（执行层）**：`verify_by_pytest` 真正运行 pytest，把测试结果结构化回传给 agent：

In [25]:
import tempfile, os
from mini_harness.harness.verifier import verify_by_pytest

def verify_by_pytest(
    test_path: str,
    cwd=None,
    timeout: int = 60,
    extra_args=None,
) -> dict:
    """
    真执行 pytest 并返回结构化结果，供 agent 作为工具调用。

    这是 Verification Loop 的核心工具：
    LLM 写完代码后调用此函数，用真实的 pytest 执行结果判断代码是否正确，
    而不是让 LLM 自己评估"代码看起来对不对"。

    Args:
        test_path: pytest 目标（文件或目录路径，如 "tests/test_main.py"）
        cwd:       工作目录，None 表示使用当前目录
        timeout:   最长等待秒数，超时后强制中止并返回失败结果
        extra_args: 追加给 pytest 的额外参数列表（如 ["--maxfail=3"]）

    Returns:
        dict，包含：
          passed:        bool，是否全部测试通过
          exit_code:     int，pytest 退出码（0=全部通过，非 0=有失败）
          summary:       str，如 "3 passed, 1 failed in 0.02s"
          stdout:        str，pytest 输出末尾 2000 字符（避免撑爆 LLM context）
          passed_count / failed_count / error_count: int，各类结果数量
    """
    # 构建 pytest 命令：
    # -v           显示每条测试用例的详细名称和结果
    # --tb=short   只显示关键的 traceback（减少输出体积）
    # --no-header  不打印 pytest 版本信息头（节省 token）
    cmd = ["python", "-m", "pytest", test_path, "-v", "--tb=short", "--no-header"]

    # 追加调用方传入的额外 pytest 参数（可选，用于精细控制测试行为）
    if extra_args:
        cmd.extend(extra_args)

    try:
        # subprocess.run 同步等待 pytest 子进程执行完毕
        # capture_output=True 捕获 stdout/stderr（不打印到终端）
        # text=True 将字节流自动解码为字符串
        # timeout 超时后抛出 subprocess.TimeoutExpired 异常
        proc = subprocess.run(
            cmd, capture_output=True, text=True, timeout=timeout, cwd=cwd
        )

        stdout = proc.stdout or ""

        # 从 stdout 末尾提取 pytest 摘要行（如 "3 passed, 1 failed in 0.02s"）
        summary = _extract_pytest_summary(stdout)

        # 从摘要行解析出各类结果的数量（passed / failed / error）
        counts = _parse_pytest_counts(summary)

        return {
            # exit_code=0 表示所有测试通过，非 0 表示有失败或错误
            "passed":       proc.returncode == 0,
            "exit_code":    proc.returncode,
            "summary":      summary,
            # 只取末尾 2000 字符：完整输出可能很长，截断后仍包含关键失败信息
            # 避免把大量日志塞进 LLM context，触发压缩或超限
            "stdout":       stdout[-2000:],
            "passed_count": counts.get("passed", 0),
            "failed_count": counts.get("failed", 0),
            "error_count":  counts.get("error", 0),
        }

    except subprocess.TimeoutExpired:
        # 超时场景：pytest 在限定时间内未完成（如存在死循环或阻塞 IO）
        # 捕获异常后返回结构化失败结果，而不是让异常向上抛出崩溃 agent
        # error_count=1 让 LLM 感知到"有问题"，触发 Verification Loop 的修复分支
        return {
            "passed":       False,
            "exit_code":    -1,                          # -1 表示非正常退出
            "summary":      f"TIMEOUT after {timeout}s",
            "stdout":       "",
            "stderr":       f"pytest timeout after {timeout}s",
            "passed_count": 0,
            "failed_count": 0,
            "error_count":  1,
        }


&emsp;&emsp;`verify_by_pytest` 用 `subprocess.run` 真正执行 pytest，返回值是结构化 dict，其中 `passed=False` + `stdout` 里的错误信息可以直接传回给 LLM，让它看到具体哪个测试失败、失败原因是什么，然后自动修复代码再重试。这是"让失败显性化"的完整闭环：<font color=red>失败 → 结构化报告 → LLM 修复 → 重新验证</font>。

### 6.1.E 验证测试

&emsp;&emsp;用两个临时 pytest 文件（一个必过、一个必败）验证 `verify_by_pytest` 能正确区分结果，并把 `passed`、`exit_code`、`passed_count`、`failed_count` 全部回传给调用方。

In [26]:
# ================================================================
# 【验证】机制 6.1：verify_by_pytest 真实执行 — 通过 vs 失败
# 纯本地，无需 LLM API，预期 3 秒内完成
# ================================================================

import os
import tempfile
from mini_harness.harness.verifier import verify_by_pytest

def print_section(title: str):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

def show_result(r: dict):
    """格式化打印验证结果"""
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  状态:         [{status}]  exit_code={r['exit_code']}")
    print(f"  摘要:         {r['summary']}")
    print(f"  passed_count: {r['passed_count']}")
    print(f"  failed_count: {r['failed_count']}")
    print(f"  error_count:  {r['error_count']}")


# ---------------------------------------------------------------
# Case 1：必然通过 — assert 1 + 1 == 2
# ---------------------------------------------------------------
print_section("Case 1：必然通过的测试")

f1 = tempfile.NamedTemporaryFile(mode="w", suffix="_test.py", delete=False)
f1.write("def test_pass():\n    assert 1 + 1 == 2\n")
f1.close()

r1 = verify_by_pytest(f1.name)
os.unlink(f1.name)

show_result(r1)


# ---------------------------------------------------------------
# Case 2：必然失败 — assert 1 + 1 == 3
# ---------------------------------------------------------------
print_section("Case 2：必然失败的测试")

f2 = tempfile.NamedTemporaryFile(mode="w", suffix="_test.py", delete=False)
f2.write("def test_fail():\n    assert 1 + 1 == 3\n")
f2.close()

r2 = verify_by_pytest(f2.name)
os.unlink(f2.name)

show_result(r2)
print(f"\n  stdout 末尾（供 LLM 定位失败原因）:")
print(f"  {r2['stdout'][-150:].strip()}")



  Case 1：必然通过的测试
  状态:         [PASS]  exit_code=0
  摘要:         ============================== 1 passed in 0.04s ===============================
  passed_count: 1
  failed_count: 0
  error_count:  0

  Case 2：必然失败的测试
  状态:         [FAIL]  exit_code=1
  摘要:         ============================== 1 failed in 0.04s ===============================
  passed_count: 0
  failed_count: 1
  error_count:  0

  stdout 末尾（供 LLM 定位失败原因）:
  t_fail - assert (1 + 1) == 3
============================== 1 failed in 0.04s ===============================


&emsp;&emsp;**关键观察**：`passed` 和 `exit_code` 来自 subprocess 的真实返回码，LLM 无法伪造；`passed_count` / `failed_count` 是从 pytest stdout 解析得到的，Agent 下一轮读到的"测试通过"完全取决于代码是否真的正确，而非模型自我声称。

&emsp;&emsp;你已经看到 pytest 真跑作为"真相来源"的最小实现。Anthropic 在生产级 agent 里用什么实现同样的思路？答案是 Puppeteer MCP。

### 6.2 对照：Puppeteer MCP 的生产级对等物

&emsp;&emsp;`verify_by_pytest` 的思路在生产级 agent 中有直接对等物：Anthropic 在 Effective Harnesses for Long-Running Agents（2025-11-26）中说：

> &emsp;Anthropic 用 Puppeteer MCP 做浏览器真机验证——agent 生成 UI 代码后，MCP 服务器启动浏览器，截图返回给 LLM，LLM 看到渲染结果再决定是否修复。失败的表现是截图里的 UI 异常，而不是 LLM 自说自话"看起来应该对"。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Verification Loop 两层 × Mini Harness vs 生产级对比</font></p>
<div class="center">

| 验证层 | Mini Harness | Anthropic 生产级 |
|--------|--------------|-----------------|
| 引导层 | `SYSTEM_PROMPT_WITH_VERIFICATION` 自我检查清单 | 同机制 system prompt |
| 执行层 | `subprocess` 真跑 pytest，回传 exit_code + stdout | Puppeteer MCP 浏览器真机验证，回传截图 |
| 失败处理 | 结构化 dict → LLM 自动修复代码 | 截图 + DOM 状态 → LLM 自动修复 UI |

</div>

&emsp;&emsp;两者的本质思路相同：<font color=red>用外部可观察的工具结果作为验证标准，而不是依赖 LLM 的自我评估</font>。Anthropic 的原话值得记住："It is unacceptable to remove or edit tests."——测试是真相的来源，不能被 agent 为了"让测试通过"而删除或修改。

&emsp;&emsp;**你现在掌握了**：用 pytest/playwright 等外部可观察工具做验证；拒绝 LLM 自我评估的循环论证；理解"测试是真相来源"的工程纪律。接下来看机制⑦ Subagents——怎么把一个复杂任务拆成多个独立工作的子 agent。

---

## <center>第七章：机制⑦ Subagents + 父子叠加预算警示</center>

&emsp;&emsp;到目前为止，你的 agent 都是单线程的：一个 Loop，一个 messages 列表，一条任务线。但现实中的复杂任务往往需要并行推进——比如"同时分析 10 个文件"或者"让一个子任务专门负责单元测试"。机制⑦ Subagents 的答案是：<font color=red>为每个子任务创建一个独立的 agent 实例，拥有自己的 messages 列表和 context</font>，父 agent 只看子任务的结果摘要，不被细节淹没。


### 为什么需要 Subagents？

单 agent 面对复杂任务时有两个根本性瓶颈：

**瓶颈一：context 污染。** 子任务的中间过程（工具调用、错误重试、临时结果）会持续写入 messages 列表。任务越复杂，context 越臃肿，LLM 的注意力被稀释，推理质量下降。

**瓶颈二：串行效率低。** 单线程 agent 必须依次完成每个步骤。若任务之间没有强依赖关系，让它们并行执行可以大幅缩短总耗时。

Subagents 通过"隔离"解决前者，通过"并发"解决后者。

### 父子 Agent 的职责划分

**父 Agent 负责：**
- 接收用户目标，规划子任务清单
- 为每个子任务创建独立的 agent 实例并分发
- 汇总各子任务返回的结果摘要
- 生成最终答案

**子 Agent 负责：**
- 在自己独立的 messages 列表中完成单一子任务
- 只将结果摘要返回给父 agent，不暴露内部细节
- 子任务完成后，其 context 自动释放，不污染父 agent

### 与单 Agent 的对比

<div style="display: flex; justify-content: center;">

| 维度 | 单 Agent | Subagents |
|:----:|:----:|:----:|
| context 管理 | 所有子任务共享一个 messages 列表 | 每个子任务拥有独立 context |
| 执行方式 | 串行，逐步完成 | 可并行，同时推进多个子任务 |
| 父级可见信息 | 全部中间过程 | 仅子任务结果摘要 |
| 适用场景 | 线性、依赖关系强的任务 | 可分解、独立性强的任务 |

</div>

### 核心设计原则

子 agent 的 context 对父 agent 是不透明的。父 agent 只需要知道"子任务成功还是失败、结果是什么"，而不需要知道子任务经历了多少次工具调用、中间出过什么错。这种信息隔离让父 agent 的 context 始终保持精简，即使管理十几个并行子任务，也不会因为细节积累而失控。

### 7.1 delegate 函数：隔离的核心

&emsp;&emsp;Subagents 要解决"父 agent 不被子任务细节淹没"的问题，核心抓手就是一个叫 `delegate` 的函数——它负责把子任务从父 agent 的 context 中隔离出去，给子 agent 一块独立的"工作空间"。这个函数长什么样？它接受哪些参数？又是如何保证隔离的？下面你直接看 `subagent.py` 中 `delegate` 的完整实现，重点关注它和 `run_agent` 的差异：`delegate` 传给子 agent 的是一个**全新构造的 prompt 字符串**，而不是父 agent 的 `messages` 列表——这是隔离的关键点。

In [27]:
def delegate(
    goal: str,
    context: str = "",
    tools=None,
    available_tools=None,
    config=None,
) -> str:
    """
    将子任务委派给一个独立的子 agent 执行。

    隔离保证：
      - 子 agent 有全新的空 messages 列表（不继承父 context）
      - 子 agent 只获得指定的工具子集（通过 tools 参数过滤）
      - 父 agent 只收到子 agent 的最终答案摘要，不是整个 messages 历史

    Args:
        goal: 子任务描述
        context: 可选的背景信息（只传子任务需要的部分，不传全部历史）
        tools: 允许子 agent 使用的工具名列表（None=全部）
        available_tools: 父 agent 的全量工具字典
        config: HarnessConfig（None 时从环境变量加载）

    Returns:
        JSON 字符串，包含 success / answer / steps / errors
    """
    if config is None:
        config = HarnessConfig.from_env()

    # 工具过滤：只给子 agent 它需要的工具
    if tools is not None:
        subagent_tools = {
            name: func for name, func in available_tools.items() if name in tools
        }
    else:
        subagent_tools = available_tools or {}

    # 构建子任务 prompt（只包含 goal + 必要 context，不是父 messages 全文）
    prompt = f"Subtask: {goal}"
    if context:
        prompt += f"\n\nContext:\n{context}"

    # 递归调用 run_agent，但用全新的 messages（隔离！）
    result = run_agent(
        user_goal=prompt,
        tools=subagent_tools,
        config=config,
        system_prompt="You are a subagent helping with a specific subtask. Focus on the given goal.",
    )
    # 只返回最终答案，不返回完整 messages 历史
    return json.dumps({"success": True, "answer": result["answer"],
                       "steps": result["steps"], "errors": result["errors"]})

&emsp;&emsp;`delegate` 的关键在于：调用 `run_agent` 时传入的是全新构建的 `prompt` 字符串，而不是父 agent 的 `messages` 列表。子 agent 从这个 `prompt` 开始自己的 Loop，父 agent 等待结果，只拿到 `answer` 字符串——不是子 agent 内部的推理过程。这就实现了 context 隔离：<font color=red>父 agent 的大脑不被子任务的细节污染</font>。

### 7.1.E 验证测试（mock）

&emsp;&emsp;本节用 mock LLM 客户端演示 subagent 机制的消息隔离结构，不演示真实模型推理质量。通过 `patch` 替换 `harness.subagent.run_agent`，让真实 `delegate` 函数运行，但不真调 API。

In [28]:
# ================================================================
# 【验证-MOCK】机制 7.1：父子 agent messages 完全隔离
# 用 Mock 替换真实 LLM 调用，无需 API Key，预期 1 秒内完成
# 真实运行效果见第十章 E2E demo
# ================================================================

import json
from unittest.mock import patch
from mini_harness.harness.subagent import delegate
from mini_harness.harness.config import HarnessConfig

def print_section(title: str):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

# 初始化配置（fake-key 仅占位，不发真实请求）
config = HarnessConfig(api_key="fake-key", base_url="http://fake")

# 模拟父 agent 已有 5 条历史消息（代表父 agent 当前的 context）
parent_messages = [
    {"role": "user",      "content": "分析这批日志"},
    {"role": "assistant", "content": "我来分析..."},
    {"role": "user",      "content": "继续"},
    {"role": "assistant", "content": "正在处理第 2 批..."},
    {"role": "user",      "content": "还有第 3 批"},
]

# ---------------------------------------------------------------
# Case 1：调用 delegate 前后，父 messages 长度不变（隔离硬证据）
# 核心断言：子 agent 运行不会向父 messages 写入任何内容
# ---------------------------------------------------------------
print_section("Case 1：父 messages 在 delegate 调用前后保持不变")

def fake_run_agent(user_goal, tools, config, system_prompt=None, hooks=None, budget=None):
    """Mock 子 agent：不调用 LLM，直接返回固定结构"""
    return {
        "answer":        f"[Mock 子 agent 完成] 收到 prompt 长度={len(user_goal)} chars",
        "steps":         3,
        "tokens":        500,
        "errors":        [],
        "budget_report": None,
        "blocked_tools": 0,
    }

len_before = len(parent_messages)
print(f"  delegate 调用前  父 messages 长度: {len_before}")

with patch("mini_harness.harness.subagent.run_agent", side_effect=fake_run_agent):
    result_json = delegate(
        goal="提取日志中的 ERROR 条目",
        context="2024-01-01 ERROR disk full\n2024-01-01 INFO startup ok",
        tools=["read"],
        available_tools={"read": lambda x: "ok"},
        config=config,
    )

len_after = len(parent_messages)
result    = json.loads(result_json)
isolated  = len_before == len_after

print(f"  delegate 调用后  父 messages 长度: {len_after}")
print(f"  隔离验证:        {'通过' if isolated else '失败'}  ({len_before} == {len_after})")
print(f"  子任务返回:      success={result['success']}  steps={result['steps']}")
print(f"  子 agent 答案:   {result['answer']}")


# ---------------------------------------------------------------
# Case 2：验证子 agent 收到的 prompt 不含父 messages 历史
# 核心断言：子 agent 从全新起点开始，看不到父 agent 的对话历史
# ---------------------------------------------------------------
print_section("Case 2：子 agent 的 prompt 不含父 messages 历史")

captured = {}

def fake_capture(user_goal, tools, config, system_prompt=None, hooks=None, budget=None):
    """Mock 子 agent：捕获收到的 user_goal，供事后验证"""
    captured["user_goal"] = user_goal
    return {"answer": "done", "steps": 1, "tokens": 100, "errors": [],
            "budget_report": None, "blocked_tools": 0}

with patch("mini_harness.harness.subagent.run_agent", side_effect=fake_capture):
    delegate(
        goal="统计 ERROR 数量",
        context="日志内容片段...",
        available_tools={},
        config=config,
    )

contains_parent_history = "分析这批日志" in captured["user_goal"]
print(f"  子 agent 收到的 prompt:")
for line in captured["user_goal"].splitlines():
    print(f"    {line}")
print(f"\n  父历史内容'分析这批日志'是否出现在子 prompt 中: "
      f"{'是 ← 隔离失败' if contains_parent_history else '否  隔离正常'}")



  Case 1：父 messages 在 delegate 调用前后保持不变
  delegate 调用前  父 messages 长度: 5
  delegate 调用后  父 messages 长度: 5
  隔离验证:        通过  (5 == 5)
  子任务返回:      success=True  steps=3
  子 agent 答案:   [Mock 子 agent 完成] 收到 prompt 长度=88 chars

  Case 2：子 agent 的 prompt 不含父 messages 历史
  子 agent 收到的 prompt:
    Subtask: 统计 ERROR 数量
    
    Context:
    日志内容片段...

  父历史内容'分析这批日志'是否出现在子 prompt 中: 否  隔离正常


&emsp;&emsp;**关键观察**：Case 1 的"5 → 5"是隔离的硬证据——`delegate` 不向父 messages 追加任何内容，子 agent 在独立空间运行。Case 2 显示子 agent 收到的是以 `Subtask:` 开头的全新 prompt 字符串，父 agent 5 条历史完全不可见。真实 LLM 跑出来会产生真实推理和 token 消耗，mock 演示的是消息隔离的架构结构。真实 LLM 调用看第十章 E2E demo（包含真实 token 消耗 + 真实输出）。

&emsp;&emsp;隔离的好处看到了。但隔离不是免费的——它的代价藏在一个反直觉的算式里：**父子 iteration budget 不是加法，是乘法**。

### 7.2 父子叠加预算：不是加法，是乘法

&emsp;&emsp;Subagents 最大的危险不是"功能无法实现"，而是"成本意外爆炸"。理解这个危险，你需要看一个计算：

&emsp;&emsp;假设父 agent 的 `max_steps=50`，每步都 spawn 一个子 agent（子 agent `max_steps=50`）：

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231721948.png" width=50% alt="父子叠加预算爆炸警示">
</div>

> **【踩坑预警】** 父子 iteration budget 是乘法关系，不是加法。很多人看到"父 50 步 + 子 50 步 = 100 步"然后放心了，实际上是 50 × 50 = 2500 步的最坏情况。这是 Manus 早期版本的真实教训（来源：[Phil Schmid，2026-01-05](https://www.philschmid.de/agent-harness-2026)）。

&emsp;&emsp;针对这个问题，有三条子 agent 设计铁律：

&emsp;&emsp;**铁律一**：子 agent 必须有独立且比父更小的**迭代步数上限**（Iteration Budget，即最多允许执行多少轮工具调用）。推荐比例：父 agent 上限 50 步，子 agent 上限 10 步。之所以是乘法关系：1 个父 agent 委派 5 个子 agent，若每个子 agent 上限也是 50 步，则整体最坏情况是 250 步，成本和时间都可能失控。

&emsp;&emsp;**铁律二**：子 agent 必须设置**结果长度上限**（Summary Cap，即子任务返回给父 agent 的内容最多允许多少字符）。子任务内部可能产生大量工具调用日志和中间结果，若不截断直接返回，会大量占用父 agent 的 context 窗口，引发 context 污染。

&emsp;&emsp;**铁律三**：子任务必须设置**成本上限**（Token Budget 或 USD 预算上限），一旦达到阈值则强制终止。这是机制⑪ Token Budget 在 Subagents 场景中的直接应用——没有成本刹车的子 agent 可能因意外的循环调用而产生高额费用。

&emsp;&emsp;Mini Harness 的 `HarnessConfig.max_steps=50` 提供了基础刹车，但对 Subagents 场景来说还不够。你需要为子 agent 单独创建一个更小的配置对象（如 `max_steps=10`），并在调用 `delegate` 时通过 `config` 参数传入，确保父子之间的步数上限互相独立、不共享。

&emsp;&emsp;**你现在掌握了**：用 `delegate` 函数将子任务从父 context 中隔离；理解父子迭代步数是**乘法关系而非加法关系**；为子 agent 单独配置更小的迭代上限和返回长度限制，构建两道防线。接下来看机制⑧ Generator-Evaluator（生成-评审双 agent）——把"生成内容"和"审查内容"分配给两个独立的 LLM 实例，用架构消除 LLM 自我审查时的固有偏差。

---

## <center>第八章：机制⑧ Generator-Evaluator（压轴震撼）</center>

&emsp;&emsp;第七章解决了单线程 loop 应对复杂任务的问题，但 Subagents 带来了新的隐患：子任务生成的结果谁来审？LLM 自审等于作者自审，天然有偏差——这就是故障模式⑦（缺自动化评审）的来源。机制⑧ Generator-Evaluator 是它的答案。

&emsp;&emsp;在全部 11 个机制里，Generator-Evaluator 是架构最优雅的一个，也是效果最难被"第一直觉"预测的一个。故障模式⑦（缺自动化评审）的本质是：单个 agent 生成结果后自我验证，等同于让作者给自己的文章打分——天然有偏差。Generator-Evaluator 的解法是：<font color=red>生成和评估分离，用独立 context 的另一个 LLM 做客观审查</font>。

### 为什么 LLM 自审是不可靠的？

LLM 在生成内容时，其注意力已经被当前的推理路径所"锚定"。当被要求"检查自己的输出"时，它倾向于沿着相同的推理路径再走一遍，而不是真正以批判的视角重新审视。这不是模型的缺陷，而是 Transformer 架构在 context 共享下的固有特性：**生成者和审查者读的是同一份 messages 历史，思维起点相同，盲点也相同**。

### 双角色分离的工作方式

Generator-Evaluator 将一次任务拆分为两个独立阶段：

**生成阶段（Generator）：**
- 接收任务目标，专注于生成高质量的内容或代码
- 不需要考虑"我写的对不对"，只需要"写出最好的答案"
- 完成后将结果传给 Evaluator

**评估阶段（Evaluator）：**
- 接收 Generator 的输出，以及原始任务目标
- 拥有完全独立的空 messages 列表，没有 Generator 的推理历史
- 从零开始审查，不受 Generator 思维路径的影响
- 返回结构化评审意见：是否通过、问题所在、改进建议

### 与第六章 Verification Loop 的关系

<div style="display: flex; justify-content: center;">

| 维度 | 机制⑥ Verification Loop | 机制⑧ Generator-Evaluator |
|:----:|:----:|:----:|
| 验证主体 | 外部工具（pytest、linter） | 另一个 LLM 实例 |
| 适用场景 | 有客观标准的验证（代码是否跑通） | 需要主观判断的评审（内容质量、逻辑严密性） |
| 评审依据 | 工具返回的确定性结果 | LLM 的推理能力 |
| context 隔离 | 不涉及 | 核心设计，Evaluator 看不到 Generator 的历史 |

</div>

两者可以组合使用：先用 Verification Loop 做客观验证，再用 Generator-Evaluator 做主观质量评审，构成双重质量保障。

### 核心设计原则

Generator-Evaluator 的价值不在于"两次 LLM 调用"，而在于 **context 隔离带来的视角独立性**。如果 Evaluator 共享 Generator 的 messages 历史，它的评审质量会显著退化——因为它看到了生成者的推理过程，会不自觉地为其辩护。真正有效的审查，来自于一个对生成过程一无所知的独立判断者。

### 8.1 角色分离原理

&emsp;&emsp;`evaluator.py` 的 `evaluate` 函数实现了 Evaluator 角色：

In [29]:
def evaluate(
    candidates: list,
    rubric: str = "correctness",
    config=None,
) -> dict:
    """
    用独立 LLM（Evaluator）评审多个候选方案，返回最优方案。

    Evaluator 有自己独立的 system prompt 和 messages 列表——
    它不知道 Generator 的推理过程，只看最终候选方案，
    这就是"独立 context"的核心价值：避免 Generator 的思路影响评审。

    Args:
        candidates: 候选方案字符串列表（Generator 的输出）
        rubric:     评审标准，如 "correctness" / "efficiency" / "clarity"
        config:     HarnessConfig（None 时从环境变量加载）

    Returns:
        dict，包含：
          best_index: int，最优候选的下标
          scores:     list[float]，每个候选的 0~1 评分
          reasoning:  str，评审理由
    """
    if config is None:
        config = HarnessConfig.from_env()

    # 将所有候选方案拼入一个 prompt：
    # Evaluator 只看候选结果，不知道 Generator 是怎么得到它们的
    prompt = f"Evaluate the following {len(candidates)} candidate solutions based on: {rubric}\n\n"
    for i, candidate in enumerate(candidates):
        prompt += f"Candidate {i + 1}:\n{candidate}\n\n"

    # 要求返回 JSON，方便后续结构化解析
    prompt += (
        f"Please evaluate each candidate on a scale of 0-1 based on {rubric}.\n"
        "Return your evaluation in JSON format:\n"
        '{"scores": [score1, score2, ...], "best_index": index_of_best, "reasoning": "explanation"}'
    )

    # 直接用 OpenAI client，不走 run_agent ——
    # Evaluator 是一次性调用，不需要工具循环，用原始接口最轻量
    client = OpenAI(api_key=config.api_key, base_url=config.base_url)
    response = client.chat.completions.create(
        model=config.model_name,
        messages=[
            # Evaluator 专属 system prompt：强调客观性和 JSON 格式输出
            # 注意：这里没有继承 Generator 的任何历史 messages，context 完全独立
            {"role": "system", "content": "You are an expert evaluator. Provide objective assessments in JSON format."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.3,  # 低温度：评审追求一致性和可重复性，不需要创意多样性
    )

    result_text = response.choices[0].message.content

    try:
        # 正常路径：LLM 按要求返回了合法 JSON
        result = json.loads(result_text)
        return {
            "best_index": result.get("best_index", 0),
            "scores":     result.get("scores", [0.5] * len(candidates)),
            "reasoning":  result.get("reasoning", ""),
        }
    except json.JSONDecodeError:
        # 降级路径：LLM 返回了非 JSON 内容（如加了 markdown 代码块）
        # 保守策略：默认选第一个候选，评分均设 0.5（中立），记录原始输出供调试
        return {
            "best_index": 0,
            "scores":     [0.5] * len(candidates),
            "reasoning":  f"JSON 解析失败，原始输出: {result_text[:200]}",
        }


&emsp;&emsp;注意两个设计细节：第一，Evaluator 的 `temperature=0.3`——评审需要稳定性和一致性，而不是创意，低温度保证同样的候选方案每次评审得分接近；第二，Evaluator 的 system prompt 只说"你是客观评审员"，它不知道 Generator 的内部推理过程，只看最终输出——这是独立 context 的关键。

### 8.1.E 验证测试（mock）

&emsp;&emsp;本节用 mock LLM 客户端演示 Generator-Evaluator 的接口结构与角色独立性，不演示真实模型推理质量。通过捕获 evaluate 传给 LLM 的 messages 和 temperature，让你看到 Evaluator 的 system prompt 与 Generator 完全不同——这是"角色隔离"的代码层证据。真实 LLM 调用看第十章 E2E demo（包含真实 token 消耗 + 真实输出）。

In [30]:
# ================================================================
# 【验证-MOCK】机制 8.1：evaluate 接口结构 + Evaluator 角色独立性
# 用 Mock 替换真实 LLM 调用，无需 API Key，预期 1 秒内完成
# 真实运行效果见第十章 E2E demo
# ================================================================

import json
from unittest.mock import patch
from types import SimpleNamespace
from mini_harness.harness.evaluator import evaluate
from mini_harness.harness.config import HarnessConfig

def print_section(title: str):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

# ── Mock 工具类 ──────────────────────────────────────────────────

def make_fake_eval_response(scores, best_index, reasoning="Mock 演示评分"):
    """构造与真实 OpenAI 响应结构完全一致的 Mock 对象"""
    json_str = json.dumps({"scores": scores, "best_index": best_index, "reasoning": reasoning})
    return SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content=json_str))]
    )

class FakeEvaluatorClient:
    """Mock OpenAI 客户端：捕获调用参数，返回预设评分结果"""
    def __init__(self, scores, best_index):
        self.captured_messages    = []   # 捕获传给 LLM 的 messages 列表
        self.captured_temperature = None # 捕获 temperature 参数
        self._scores     = scores
        self._best_index = best_index
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, model, messages, temperature, **kwargs):
        self.captured_messages    = messages
        self.captured_temperature = temperature
        return make_fake_eval_response(self._scores, self._best_index)

# ── 准备测试数据 ─────────────────────────────────────────────────

candidates = [
    "方案 A：用 dict 存储结果",
    "方案 B：用 dataclass 存储结果",
    "方案 C：用 namedtuple 存储结果",
]
fake_client = FakeEvaluatorClient(scores=[0.6, 0.85, 0.4], best_index=1)

with patch("mini_harness.harness.evaluator.OpenAI", return_value=fake_client):
    result = evaluate(candidates, rubric="代码可读性")

# ---------------------------------------------------------------
# Case 1：接口返回结构验证，evaluate 函数的输出格式是否符合预期
# ---------------------------------------------------------------
print_section("Case 1：接口返回结构验证")

best = candidates[result["best_index"]]
print(f"  best_index : {result['best_index']}  →  {best}")
print(f"  scores     : {result['scores']}")
for i, (candidate, score) in enumerate(zip(candidates, result["scores"])):
    marker = " ←最优" if i == result["best_index"] else ""
    print(f"    [{i}] score={score:.2f}  {candidate}{marker}")
print(f"  reasoning  : {result['reasoning']}")

# ---------------------------------------------------------------
# Case 2：Evaluator 角色独立性验证
# 核心断言：Evaluator 的 messages[0] 是独立的 system prompt，
# 不包含 Generator 的任何推理历史
# ---------------------------------------------------------------
print_section("Case 2：Evaluator 角色独立性（system prompt 隔离）")

sys_msg = fake_client.captured_messages[0]
print(f"  messages 总长度 : {len(fake_client.captured_messages)} 条（system + user，无 Generator 历史）")
print(f"  messages[0] role    : {sys_msg['role']}")
print(f"  messages[0] content : {sys_msg['content']}")

# ---------------------------------------------------------------
# Case 3：temperature 硬约束验证
# 评审场景要求低温度，确保结果一致可重复
# ---------------------------------------------------------------
print_section("Case 3：temperature 硬约束验证（评审要求 0.3）")

t = fake_client.captured_temperature
status = "通过" if t == 0.3 else "失败"
print(f"  captured temperature : {t}")
print(f"  temperature == 0.3   : {status}")



  Case 1：接口返回结构验证
  best_index : 1  →  方案 B：用 dataclass 存储结果
  scores     : [0.6, 0.85, 0.4]
    [0] score=0.60  方案 A：用 dict 存储结果
    [1] score=0.85  方案 B：用 dataclass 存储结果 ←最优
    [2] score=0.40  方案 C：用 namedtuple 存储结果
  reasoning  : Mock 演示评分

  Case 2：Evaluator 角色独立性（system prompt 隔离）
  messages 总长度 : 2 条（system + user，无 Generator 历史）
  messages[0] role    : system
  messages[0] content : You are an expert evaluator. Provide objective assessments in JSON format.

  Case 3：temperature 硬约束验证（评审要求 0.3）
  captured temperature : 0.3
  temperature == 0.3   : 通过


&emsp;&emsp;**关键观察**：Case 1 显示 evaluate 返回标准三字段结构（best_index/scores/reasoning），与 Generator 输出完全解耦。Case 2 显示 Evaluator 的 system prompt 是独立的"expert evaluator"角色定义，与 Generator 使用的 system prompt 完全不同——这是角色隔离的直接代码证据。Case 3 确认 temperature=0.3 被硬编码传入，保证评审结果的稳定性；真实 LLM 跑出来会对三个候选给出有意义的分差，mock 演示的是接口结构本身而非模型推理质量。

&emsp;&emsp;单次 `evaluate` 的接口结构和角色独立性你已经看清楚了。但 Generator-Evaluator 的真正价值不在单次评分，而在迭代循环——下面看完整流程。

### 8.2 Generator-Evaluator Loop 的完整流程

&emsp;&emsp;单次 `evaluate` 调用展示了 Evaluator 的评审能力，但 Generator-Evaluator 的真正价值在循环里。一个完整的 Generator-Evaluator Loop 是这样运作的：

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231638887.png" width=50% alt="Generator-Evaluator Loop 选代示意图">
</div>

&emsp;&emsp;每次 Generator 生成时，上一轮的评审意见被追加到它的 context 里，让它"知道哪里不好"并针对性改进。Evaluator 的评分从 0.6 升到 0.92，这个提升来自 Generator 的迭代修正——而不是 Evaluator 标准的放宽。这里借用了 GAN 的思想——一方生成、一方挑刺、循环到达标。Generator / Evaluator / Orchestrator 三角分工是本课对这一模式的教学抽象；Anthropic 博客《[Harness Design for Long-Running Apps](https://www.anthropic.com/engineering/harness-design-long-running-apps)》（2026-03-24）讨论了类似的多 Agent 协作思路，但并未以此命名框架，GAN 类比亦为本课自行引入（GAN 是机器学习中生成模型 vs 判别模型对抗训练的架构，你不需要懂 GAN 也能理解：一方生成、一方挑刺、循环到达标）。

### 8.3 为什么 Evaluator 可以用小模型

&emsp;&emsp;这里有一个高价值的工程洞见：<font color=red>Evaluator 不需要和 Generator 一样大的模型</font>。道理很简单——写作（生成）需要创造力，挑刺（评审）只需要判断力。一个 7B 的小模型可以准确识别"这段代码没有错误处理"，但它未必能从零生成一段优秀的错误处理代码。Evaluator 的工作是判断，不是创造。

&emsp;&emsp;这在工程上的含义是：用 Evaluator 降成本。Generator 用大模型（能力强，贵），Evaluator 用小模型（判断力够用，便宜）——整个 Loop 的成本可以显著降低，而质量不明显下降。你也可以在 `evaluate` 调用时传入一个配置了小模型的 `HarnessConfig`，比如用 `deepseek-chat` 做 Evaluator，用更大的模型做 Generator。

&emsp;&emsp;**你现在掌握了**：理解 Generator / Evaluator 分工的架构价值；知道为什么 Evaluator 可以用小模型而 Generator 不能；能把这套模式落到 `evaluate` 函数的真实 API 设计上。接下来进入最后一段生产级三扩展——⑨ Permission Gate、⑩ Hooks、⑪ Token Budget，它们是让你的 Mini Harness 真正能上生产的最后一公里。

---

## <center>第九章：Hooks六大标准事件</center>

&emsp;&emsp;如果你已经把 第二章到第八章 的机制串联起来，你的 Mini Harness 已经具备了 8 个核心机制。这三个生产级扩展是最后的保障层：让你的 agent 在真实任务中安全运行、成本可控、行为可观察。这三个机制放在一起讲，是因为它们共享一个基础设施：`HookManager`（机制⑩）。Permission Gate 通过 hook 拦截危险命令，Token Budget 通过 hook 在 session 结束时输出报告，Progress Tracker（第三章 已讲）也通过 hook 挂载。<font color=red>先理解 Hooks，才能理解 Permission 和 Budget 为什么是现在这个样子</font>。

### 9.1 机制⑩ Hooks：Harness 的神经系统

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231639235.png" width=50% alt="Hooks 事件总线架构"></p>


&emsp;&emsp;Hooks 本身不直接救任何一条故障模式，而是为 Permission Gate、Progress Tracker 和 Budget Guard 提供挂载基础设施。你可以把 `HookManager` 理解为一块电路板：Permission Gate、Progress Tracker、Budget Guard 是插上去的模块，Hooks 是它们共享的总线。没有 Hooks，这三个机制就只能把逻辑硬编码进 `run_agent` 主循环，耦合度极高、难以独立替换。

&emsp;&emsp;`hooks.py` 定义了整个事件总线。先看你需要了解的六个标准事件：

In [33]:
# 六大标准事件（Harness 的神经节点）
EVENTS = [
    "session_start",   # 会话开始：加载 Memory / 初始化 progress.md 头部
    "pre_iteration",   # 每轮 LLM 调用前：context 压缩检查、Budget 预检
    "post_iteration",  # 每轮 LLM 调用后：usage 统计、状态保存
    "pre_tool_use",    # 工具调用前（可 veto）：Permission Gate 的挂载点
    "post_tool_use",   # 工具调用后：错误显性化、lint、progress 记录
    "session_stop",    # 会话结束：final report、Budget 落账、进度汇总
]

# 可 veto（否决） 的事件（只有 pre_tool_use，其余为普通事件）
GATE_EVENTS = {"pre_tool_use"}

&emsp;&emsp;六个事件覆盖了 Agent Loop 的完整生命周期。`GATE_EVENTS` 中的 `pre_tool_use` 是唯一的"门控事件"——任何注册在这里的 handler 返回 `False` 就可以 veto 工具调用。Permission Gate 就挂在这里。

&emsp;&emsp;`HookManager` 的核心实现：

In [31]:
class HookManager:
    """
    Hook 事件管理器，维护每个事件的 handler 列表。

    两类接口：
      trigger(event, *args)：普通事件，顺序调用所有 handler，不中断流程
      trigger_gate(event, *args)：门控事件，任一 handler 返回 False 则 veto
    """

    def __init__(self):
        # 每个事件维护一个 handler 列表
        self.handlers = {e: [] for e in EVENTS}

    def register(self, event: str, fn) -> None:
        """注册 handler 到指定事件。fn 可以是任何可调用对象。"""
        if event not in self.handlers:
            raise ValueError(f"未知事件: {event}")
        self.handlers[event].append(fn)

    def trigger(self, event: str, *args, **kwargs) -> list:
        """
        触发普通事件，顺序调用所有 handler。

        失败的 handler 不影响其他 handler（容错执行）。
        Returns: 所有 handler 的返回值列表
        """
        results = []
        for fn in self.handlers.get(event, []):
            try:
                results.append(fn(*args, **kwargs))
            except Exception as e:
                # 普通 handler 失败只记录，不中断流程
                results.append({"__hook_error__": str(e)})
        return results

    def trigger_gate(self, event: str, *args, **kwargs):
        """
        触发门控事件（仅 pre_tool_use）。

        任一 handler 返回 (False, reason) 则立即 veto。
        handler 失败视为 veto（安全优先）。

        Returns:
            (True, "ok")：所有 handler 放行
            (False, reason)：某 handler 否决，reason 说明原因
        """
        if event not in GATE_EVENTS:
            raise ValueError(f"Event {event} is not a gate event. Valid: {GATE_EVENTS}")
        for fn in self.handlers.get(event, []):
            try:
                result = fn(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:
                    ok, reason = result
                    if not ok:
                        return False, str(reason)  # 立即 veto，不继续检查后续 handler
            except Exception as e:
                # gate handler 失败视为 veto（安全优先原则）
                return False, f"Gate handler {fn.__name__} 失败: {e}"
        return True, "ok"

    def clear(self, event: str = None) -> None:
        """清空指定事件的 handler（或全部事件）。测试 / 验证 cell 用它避免污染。"""
        if event is None:
            for e in self.handlers:
                self.handlers[e] = []
        elif event in self.handlers:
            self.handlers[event] = []

&emsp;&emsp;普通事件和门控事件的设计体现了两种不同的安全策略：普通事件（如 `post_tool_use`）失败了没关系，不影响主流程；门控事件（`pre_tool_use`）失败了则视为拦截，因为允许一个"门坏了"的危险命令执行比多记录一条日志严重得多。

### 9.1.E 验证测试

&emsp;&emsp;演示 HookManager 两种事件模式：普通事件多 handler 顺序执行互不干扰，gate 事件任一 handler 返回 False 即立即 veto 拦截后续调用。

In [34]:
# ================================================================
# 【验证】机制 9.1：普通事件 vs Gate Veto 机制
# 纯本地，无需 LLM API，预期 1 秒内完成
# ================================================================

def print_section(title: str):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

hooks = HookManager()

# ---------------------------------------------------------------
# Part A：普通事件（trigger）— 多个 handler 全部被依次调用
# 验证：post_tool_use 为普通事件，注册的 handler 互不干扰、全部执行
# ---------------------------------------------------------------
print_section("Part A：普通事件 — 多个 handler 全部执行")

log = []
hooks.register("post_tool_use",
    lambda *args, **kw: log.append(f"handler-A  tool={kw.get('tool_name', '?')}"))
hooks.register("post_tool_use",
    lambda *args, **kw: log.append(f"handler-B  result={kw.get('result', '?')}"))

hooks.trigger("post_tool_use", tool_name="echo", tool_args={}, result="hi")

for entry in log:
    print(f"  {entry}")
print(f"\n  共 {len(log)} 个 handler 执行  →  {'全部触发' if len(log) == 2 else '异常'}")


# ---------------------------------------------------------------
# Part B：Gate 事件（trigger_gate）— handler 返回 False 立即 veto
# 验证：pre_tool_use 为 gate 事件，handler 返回 (False, reason) 可拦截工具调用
# ---------------------------------------------------------------
print_section("Part B：Gate 事件 — handler 返回 False 触发 veto")

hooks.register("pre_tool_use",
    lambda *args, **kw: (False, "禁止使用 dangerous_tool"))

ok, reason = hooks.trigger_gate("pre_tool_use", "dangerous_tool", {})
status = "被拦截" if not ok else "放行"
print(f"  ok     : {ok}")
print(f"  reason : {reason!r}")
print(f"  结果   : 工具调用{status}  →  {'veto 生效' if not ok else '未拦截（异常）'}")


# ---------------------------------------------------------------
# Part C：边界验证 — 非 gate 事件调用 trigger_gate 应抛 ValueError
# 验证：框架对事件类型做了保护，防止误用 trigger_gate 处理普通事件
# ---------------------------------------------------------------
print_section("Part C：边界保护 — 非 gate 事件调用 trigger_gate")

try:
    hooks.trigger_gate("post_tool_use", "echo", {})
    print("  未抛出异常  →  边界保护失效（异常）")
except ValueError as e:
    print(f"  捕获 ValueError  →  边界保护正常")
    print(f"  错误信息: {e}")


# ---------------------------------------------------------------
# 清理：重置所有 handlers，避免污染后续 cell
# ---------------------------------------------------------------
print_section("清理")

hooks.clear()
all_empty = all(len(v) == 0 for v in hooks.handlers.values())
print(f"  所有 handlers 已清空  →  {'通过' if all_empty else '清理失败'}")



  Part A：普通事件 — 多个 handler 全部执行
  handler-A  tool=echo
  handler-B  result=hi

  共 2 个 handler 执行  →  全部触发

  Part B：Gate 事件 — handler 返回 False 触发 veto
  ok     : False
  reason : '禁止使用 dangerous_tool'
  结果   : 工具调用被拦截  →  veto 生效

  Part C：边界保护 — 非 gate 事件调用 trigger_gate
  捕获 ValueError  →  边界保护正常
  错误信息: Event post_tool_use is not a gate event. Valid: {'pre_tool_use'}

  清理
  所有 handlers 已清空  →  通过


&emsp;&emsp;**关键观察**：Part A 两个 handler 顺序执行且都出现在 log 中，证明普通事件不可被单个 handler 中止；Part B `ok=False` 证明 gate 机制一票否决，这正是 Permission Gate 能挂在 `pre_tool_use` 拦截危险命令的基础；Part C 的 `ValueError` 说明普通事件与 gate 事件接口不可混用，防止误用绕过安全检查。


---

## <center>第十章：完整跑通 + Baseline vs Full Harness 真实数据</center>

&emsp;&emsp;前面第二章到第九章，你一一见过每个机制的代码。现在是把它们串联起来、看整体效果的时刻。E2E demo 脚本 `scripts/run_e2e_demo.py` 就是这个串联点——它同时运行 Baseline（无 Harness）和 Full Harness（8 机制协同），让你直接对比两者的差异。

&emsp;&emsp;说"8 机制"而非 11，是因为 demo 脚本只接入了 ①Agent Loop / ②Tool Use / ③Progress / ⑤Feature List / ⑥Verification / ⑨Permission Gate / ⑩Hooks / ⑪Token Budget 这 8 个机制；④Context Management（第四章）、⑦Subagents（第七章）、⑧Generator-Evaluator（第八章）是横向能力，已在各自章节单独演示，E2E demo 里不重复 import，保持脚本最小可运行。

### 10.1 真实 E2E 对比数据

&emsp;&emsp;以下是 2026-04-22 真实跑通的数据（模型：`deepseek-chat`，任务：分析 `main.py` 的重复代码并给出重构建议）：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Baseline vs Full Harness 端到端真实对比（2026-04-22 / deepseek-chat）</font></p>
<div class="center">

| 维度 | Baseline（单轮调用） | Full Harness（8 机制 demo） | 倍率 |
|------|---------------------|------------------------|------|
| 步骤数 | 1 | 7 | 7× |
| 耗时 | 35.45 s | 94.02 s | 2.65× |
| 总 tokens | 1104 | 15053 | 13.6× |
| 答案字符数 | 2498 | 2819 | +13% |
| 成本 (USD) | — | $0.0697（保守估算，见上文括注） | — |
| 预算上限 | — | $0.50 | — |
| Permission 拦截 | 0 | 0（本次无危险命令） | — |
| 错误数 | 0 | 0 | — |
| progress.md | 无 | 已落盘（7 条 session 记录） | — |

</div>

&emsp;&emsp;这两个数字是本节的核心结论：**13.6× tokens，2.65× 耗时**。这不是 Harness 效率低，而是 Harness 的成本结构。你用 13 倍的 tokens 换来了什么？

### 10.2 三性：可观察性 / 可干预性 / 可控性

&emsp;&emsp;<font color=red>Full Harness 用 13× tokens 和 2.65× 耗时换来的，是三个用单轮调用永远买不到的属性</font>：

&emsp;&emsp;**可观察性**：`progress.md` 落盘了 7 条记录，每一步做了什么、调用了哪个工具、结果是什么，全程可追溯。Baseline 跑完了，你只能看到最终答案，不知道它"想了什么"。

&emsp;&emsp;**可干预性**：Permission Gate 就绪，即使这次任务没有危险命令，下次任何危险命令都会被在执行前拦截。Baseline 里 agent 直接执行任何工具，包括 `rm -rf`——你没有机会干预。

&emsp;&emsp;**可控性**：Token Budget 守住了整个任务，$0.0697 < $0.50 上限，`tripped=False`。如果任务出现异常循环，Budget 会触发优雅终止而不是无限烧钱。Baseline 的单轮调用本身就只有一次，但在真实的多步任务里，没有 Budget 的 agent 可能烧掉 10 倍于你预期的成本。

> &emsp;Anthropic 在 2026-03-24 的博客《Harness design for long-running application development》中坦率承认："Every component in a harness encodes an assumption about what the model can't do on its own, and those assumptions are worth stress testing, both because they may be incorrect, and because they can quickly go stale as models improve."
> （来源：[Anthropic · Harness Design for Long-Running Apps](https://www.anthropic.com/engineering/harness-design-long-running-apps)）

&emsp;&emsp;这句话很重要：Harness 的每个组件都在编码一个假设——关于模型做不到什么。随着模型能力提升，这些假设会慢慢过时（go stale）。今天你为 agent 加了 Permission Gate，是因为模型可能误执行危险命令；如果未来的模型在默认情况下就能做到"不执行危险命令"，这个组件就可以从 Harness 里移除。<font color=red>这不是劝退，是让你建立正确的成本直觉：Harness 是工具，不是信仰</font>。

### 10.3 Mini Harness vs 生产级放大倍率

<p align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260426231643657.png" width=50% alt="Mini Harness 与生产级工具对比"></p>


&emsp;&emsp;Mini Harness 的 11 个机制和生产级 agent（Claude Code / OpenAI Codex）的对应关系：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Mini Harness vs 生产级工具对照（功能放大关系）</font></p>
<div class="center">

| 机制 | Mini Harness | Claude Code 生产版 | OpenAI Codex 生产版 |
|------|-------------|-------------------|-------------------|
| Agent Loop | `while` + `max_steps=50` | 多层事件驱动 | App Server JSON-RPC |
| Context | `head(2)+tail(6)+middle summary` | 5 层 compaction pipeline | AGENTS.md 渐进披露 |
| Progress | `progress.md` 追加写 | `claude-progress.txt` + git | 待补充 |
| Feature List | `todo_tool` JSON 状态机 | 200+ 项结构化清单 | PLANS.md 持久化 |
| Verification | pytest subprocess | Puppeteer MCP 真机 | structural test |
| Subagents | `delegate` 递归隔离 | `/agent` + 独立 workflow | 待补充 |
| Permission | 正则黑名单 14 条 | 27 个 hook + 规则矩阵 | sandbox policy |
| Hooks | `HookManager` 6 事件 | Claude Code 27 事件 | middleware 3 hook |
| Budget | `BudgetGuard` + `BudgetExceeded` | token budget + iteration cap | per-model quota |

</div>

&emsp;&emsp;每一行你都能看到 Mini Harness 和生产工具的对应关系。Mini Harness 是"裸骨架"——功能极简，但结构透明；生产工具是成熟系统——功能丰富，但内部封装。你今天学的是骨架，以后读生产工具的源码时，你会认出每一块肌肉。

### 10.4 如何运行 E2E Demo

&emsp;&emsp;你可以自己重现这个 E2E 数据。确保 `.env` 配置好后，在 `HarnessEngineering/lesson2/` 目录下运行：

In [35]:
# 确保你当前在 HarnessEngineering/lesson2/ 目录下
!cd mini_harness && python scripts/run_e2e_demo.py

端到端 Demo：Baseline vs Full Harness（10 机制）
模型：deepseek-chat
测试文件：/Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engineering 基础入门/课件资料/mini_harness/test_data/refactor_project/main.py
模式：both

[Baseline Agent] 单轮 LLM 调用，无 Harness 机制
  steps=1  elapsed=9.35s  tokens=983
  → saved /Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engineering 基础入门/课件资料/demo_outputs/baseline_result.json

[Full Harness] 启用全部 10 机制（6 原有 + 4 新增）
  steps=9  elapsed=43.68s  tokens=29139  blocked=0  errors=0
  budget: cost=$0.1245 / max=$0.50 tripped=False
  progress.md 写入：/Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engineering 基础入门/课件资料/demo_outputs/progress.md
  → saved /Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engineering 基础入门/课件资料/demo_outputs/harness_result.json

✅ E2E Demo 完成
  JSON 报告：/Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engineering 基础入门/课件资料/demo_outputs/e2e_result.json
  MD 报告：/Users/mac/大模型资料/Harness Engineering驾驭工程/大模型Agent Harness Engi

&emsp;&emsp;脚本会依次运行 Baseline 和 Full Harness，在 `demo_outputs/` 目录输出 `progress.md`、`baseline_result.json`、`harness_result.json` 和 `e2e_report.md` 四个文件。`E2E_MODE=baseline` 环境变量可以只跑 Baseline，`E2E_MODE=harness` 只跑 Full Harness。

&emsp;&emsp;运行成功后，你会在 `demo_outputs/` 下看到 4 个文件。打开 `e2e_report.md`，如果 Baseline 和 Full Harness 都有完整输出、budget 报告里 `tripped=False`，说明跑通了。常见失败情况与排查方向：API Key 未配置会报 `Authentication error`，确认 `.env` 中的 Key 正确填写；网络超时可以用 `E2E_MODE=harness` 单独重跑 Full Harness 侧；`BudgetExceeded` 异常说明任务消耗超过预算上限，可以在脚本里调高 `max_usd` 参数再重试。

---

## <center>第十一章：课程总结与核心结论</center>

&emsp;&emsp;至此你已经看完 11 个机制的每一行代码，跑通了 E2E demo，见过 13.6× tokens 和 2.65× 耗时的真实数字。本章不讲新内容，只做三件事：用一张表把第十章 E2E demo 验证过的 8 机制收束起来、提炼三条能带走的核心结论、让你对自己此刻的能力有一个清晰的自检。

### 11.1 8 机制全景表

&emsp;&emsp;下面这张表是第十章 E2E demo 跑通的 8 个机制的全景对照。合上 notebook，你应该能照着这张表复述每个机制救的是哪条故障模式、对应哪个文件：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>8 核心机制全景对照（E2E demo 验证）</font></p>
<div class="center">

| 编号 | 机制 | 救的故障模式 | 核心文件 | 一句话作用 |
|------|------|------------|---------|----------|
| ① | Agent Loop | 一轮调用无法完成多步任务 | `core.py` | `while` 循环驱动模型多步推理 |
| ② | Tool Use | 模型无法直接操作外部系统 | `core.py` | `dispatch_tool` 把工具调用接到真实函数 |
| ③ | Progress Tracking | 任务中断后无法续跑 | `progress.py` | `progress.md` 追加写，全程可追溯 |
| ⑤ | Feature List | 复杂任务缺乏结构化计划 | `planner.py` | `todo_tool` 三模式状态机 |
| ⑥ | Verification Loop | 模型自评不可信 | `verifier.py` | `pytest subprocess` 用事实说话 |
| ⑨ | Permission Gate | 模型可能误执行危险命令 | `permission.py` | 14 条黑名单在执行前拦截 |
| ⑩ | Hooks | 机制扩展需要侵入主循环 | `hooks.py` | 6 事件切面，横向挂载 |
| ⑪ | Token Budget | 任务失控可能无限烧钱 | `budget.py` | `BudgetGuard` 触发优雅终止 |

</div>

&emsp;&emsp;<font color=red>备注</font>：④ Context Management（第四章）、⑦ Subagents（第七章）、⑧ Generator-Evaluator（第八章）是横向能力，已在各自章节单独演示。E2E demo 脚本为了保持最小可运行，没有把它们全部接入，但你在真实项目里会随时用到它们。

### 11.2 三条带走的核心结论

&emsp;&emsp;**结论一：Harness 的 13× tokens 换的是三性，不是低效**

&emsp;&emsp;第十章 10.1 的真实数字是 13.6× tokens、2.65× 耗时。这不是 Harness 效率低，而是它的成本结构：你用 13 倍的 tokens 买到了 Baseline 永远买不到的**可观察性、可干预性、可控性**。成本直觉建立后，你才能判断"这个任务值不值得上 Harness"——不是所有任务都需要，但凡是你需要追溯、拦截、兜底的任务，这个成本就是必要的。

&emsp;&emsp;**结论二：Harness 与底层模型解耦，换 SDK 骨架完全复用**

&emsp;&emsp;Mini Harness 基于 OpenAI Python SDK 写成，但第十章 10.3 的映射表已经证明：Harness 的每个机制都能在 Claude Code、Codex 里找到对应实现。如果你把 `core.py` 的主循环换到 Anthropic SDK，只需要动 API 调用、循环结束判断、tool call 位置、结果回写格式、tool schema 字段这 5 处——Context Management、Progress Tracking、Permission Gate 等深层机制一行代码都不需要动。<font color=red>Harness 的架构逻辑与模型提供商是完全解耦的</font>。

&emsp;&emsp;**结论三：Harness 会 go stale，它是工具不是信仰**

&emsp;&emsp;Anthropic 在 2026-03-24 的博客里坦率承认：“Every component in a harness encodes an assumption about what the model can’t do on its own”。今天你为 agent 加 Permission Gate，是因为模型可能误执行危险命令；如果未来模型默认就能做到“不执行危险命令”，这个组件就可以从 Harness 里移除。<font color=red>随着模型能力提升，Harness 的组件会慢慢过时（go stale），这不是劝退，是让你建立正确的成本直觉：Harness 是工具，不是信仰</font>。

---

## <center>附录：快速检索表</center>

&emsp;&emsp;以下是本节所有关键代码位置、术语和引用的快速检索表，方便你复习和查找：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>术语 × 代码文件 × 课件模块快速检索</font></p>
<div class="center">

| 术语 / 机制 | 核心代码文件 | 关键函数 / 类 | 课件模块 |
|------------|------------|-------------|---------|
| Agent Loop | `core.py` | `run_agent`，`while steps < config.max_steps` | 2.1 节 |
| Tool Use | `core.py` | `dispatch_tool`，`_build_tool_schemas` | 2.2 节 |
| Progress Tracking | `progress.py` | `ProgressTracker.register_to` | 3.1 节 |
| Context Management | `context.py` | `compress_if_needed`，`estimate_tokens` | 4.1 节 |
| 三铁律 | — | 不改 toolset / memory / system prompt | 4.2 节 |
| Feature List | `planner.py` | `todo_tool`，三模式（替换/合并/查询） | 5.1 节 |
| Verification Loop | `verifier.py` | `verify_by_pytest`，`SYSTEM_PROMPT_WITH_VERIFICATION` | 6.1 节 |
| Subagents | `subagent.py` | `delegate`，隔离原理 | 7.1 节 |
| Generator-Evaluator | `evaluator.py` | `evaluate`，`temperature=0.3` | 8.1 节 |
| Hooks | `hooks.py` | `HookManager`，`EVENTS`，`trigger_gate` | 9.1 节 |
| Permission Gate | `permission.py` | `PermissionGate.check`，14 条黑名单 | 9.2 节 |
| Token Budget | `budget.py` | `BudgetGuard.add`，`BudgetExceeded` | 9.3 节 |
| HarnessConfig | `config.py` | `HarnessConfig.from_env` | 1.3 节 |
| E2E Demo | `scripts/run_e2e_demo.py` | `run_baseline`，`run_full_harness` | 10.4 节 |

</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>本节权威引用清单</font></p>
<div class="center">

| 来源 | 日期 | URL | 用途 |
|------|------|-----|------|
| Anthropic · Effective Harnesses for Long-Running Agents | 2025-11-26 | https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents | 6.2 节 Verification Loop 引用 |
| Anthropic · Harness Design for Long-Running Apps | 2026-03-24 | https://www.anthropic.com/engineering/harness-design-long-running-apps | 8.2 节 Generator-Evaluator + 10.2 节 "go stale" |
| OpenAI · Unlocking the Codex harness | 2026-02-04 | https://openai.com/index/unlocking-the-codex-harness/ | 11.2 节 Responses API |
| Phil Schmid · The importance of Agent Harness in 2026 | 2026-01-05 | https://www.philschmid.de/agent-harness-2026 | 7.2 节 Manus 烧钱案例 |
| AGENTS.md | — | https://agents.md/ | 4.2 节 三铁律 |
| E2E Demo 真实数据 | 2026-04-22 | `demo_outputs/` 目录 | 10.1 节 真实数字来源 |

</div>